# Caso 6.1 · Regresión — Predicción de la potencia eléctrica de una central de ciclo combinado

**Libro:** *Metodología de Investigación Algocrática: Roiman Valbuena (2027)* · Capítulo 6 (Casos prácticos)
**Dataset:** Combined Cycle Power Plant (UCI Machine Learning Repository) · **Plataforma:** Google Colab (también corre en local)

---

## Cómo usar este cuaderno

1. Ejecuta de arriba hacia abajo (*Entorno de ejecución → Ejecutar todo*). Si el archivo no está en `/content`, la **Celda 2** abrirá el selector de archivos de Colab.
2. Sube `Folds5x2_pp.xlsx` (o el `CCPP.zip` de UCI, o un CSV equivalente con las columnas AT, V, AP, RH, PE).
3. **Cada celda de código está precedida por una celda de texto (🔎)** que explica *qué hace*, *por qué* y *qué observar en la salida*.
4. Al final se generan los **artefactos auditables** (reporte técnico, manifiesto, pre-registro, figuras, modelo) en la carpeta `caso61_salidas/`, comprimidos en un `.zip`.

## Mapa del cuaderno (Modelo de 6 pasos del Capítulo 2)

| Paso | Qué se hace | Celdas |
|---|---|---|
| — | Planteamiento y tipo de problema | (texto) |
| — | Entorno, semillas y registro de auditoría | 1 |
| **1** | Identificar el dataset y perfilarlo | 2 – 4 |
| **2** | Función objetivo, métrica y criterios **pre-registrados** | 5 |
| **3** | Preparar los datos: partición sin fuga y *test sellado* | 6 – 7 |
| **4** | Elegir arquitecturas y justificarlas | 8 |
| **5** | Entrenar y validar: CV repetida, curvas, búsqueda, CV anidada, residuos | 9 – 12 |
| **6** | Evaluación final, interpretabilidad, documentación y auditoría | 13 – 17 |
| Extra | Trampa deliberada (opcional) | 18 |

## Convenciones

- Los parámetros editables están en la **Celda 1**. `MODO_RAPIDO = True` reduce repeticiones para una *prueba de humo*; **sus resultados no son válidos para el libro**.
- Tiempo estimado con `MODO_RAPIDO = False`: unos **8 a 15 minutos** en CPU de Colab (depende del plan y de la carga; la Celda 11 es la más lenta).
- Todo número que aparece en el reporte final proviene de la ejecución; nada está escrito a mano.

# 1. Planteamiento del problema

## Contexto

Una **central de ciclo combinado** (CCPP) genera electricidad con turbinas de gas y de vapor integradas en un mismo ciclo. Su potencia neta depende de las condiciones del entorno: la temperatura y la presión del aire que ingresa a la turbina de gas, la humedad relativa y el vacío en el escape de la turbina de vapor.

El dataset de UCI (Tüfekci, 2014) reúne **9.568 registros horarios promedio** tomados durante seis años (2006–2011), con la planta operando a **plena carga**:

| Variable | Rol | Descripción | Unidad | Rango documentado |
|---|---|---|---|---|
| `AT` | Predictora | Temperatura ambiente | °C | 1,81 – 37,11 |
| `V` | Predictora | Vacío en el escape (turbina de vapor) | cm Hg | 25,36 – 81,56 |
| `AP` | Predictora | Presión ambiente | mbar | 992,89 – 1033,30 |
| `RH` | Predictora | Humedad relativa | % | 25,56 – 100,16 |
| `PE` | **Objetivo** | Potencia eléctrica neta horaria | MW | 420,26 – 495,76 |

## Pregunta de investigación

> ¿Es posible predecir la potencia neta horaria `PE` de la central, con un error de ingeniería aceptable, a partir de cuatro variables ambientales y de vacío? ¿Qué tan **estable, incierta y auditable** es esa predicción?

## Objetivos

1. **O1.** Construir un modelo predictivo de `PE`.
2. **O2.** Cuantificar su incertidumbre (intervalos de confianza, intervalos de predicción).
3. **O3.** Auditar el proceso: fuga de datos, estabilidad, residuos, sesgo.
4. **O4.** Producir un artefacto reproducible y documentado.

## Hipótesis de rendimiento (la forma algocrática de H₀/H₁)

- **H₀:** el modelo candidato **no** reduce el RMSE al menos un 20 % respecto de la mejor línea base simple.
- **H₁:** el modelo candidato **sí** lo reduce en al menos un 20 %, cumpliendo además la tolerancia de ingeniería declarada de antemano (Celda 5).

## Alcance y límites (declarados desde el inicio)

- **Un solo sitio y régimen de plena carga:** no se debe generalizar a otras plantas ni a cargas parciales.
- **El archivo no incluye marca de tiempo:** no podremos validar hacia el futuro ni medir deriva temporal. La independencia de las filas es un supuesto, no una verificación.
- **Asociación, no causalidad:** el vacío `V` pertenece al lado de la turbina de vapor y está fuertemente correlacionado con la temperatura; sus efectos no son separables sin conocimiento físico adicional.
- **No extrapolar** fuera de los rangos observados (ver Celda 4).

## Por qué importa en ingeniería

Una predicción confiable de la potencia esperada sirve como **línea base de desempeño**: una diferencia grande entre la potencia medida y la predicha es una señal temprana de degradación o falla. El interés no es solo predecir bien, sino saber **cuánto confiar** en la predicción.

# 2. ¿Por qué es un problema de **regresión** y no de clasificación ni de clustering?

Recuerda que en 1.1.8. te indico cómo elegir el tipo de problema que abordarás. La elección del tipo de problema no es una preferencia: se **deduce** de la pregunta y de la naturaleza de los datos.

## Preguntas guía

| # | Pregunta | Respuesta en este caso | Consecuencia |
|---|---|---|---|
| 1 | ¿Existe una variable objetivo **medida**? | Sí: `PE` (MW) | Aprendizaje **supervisado** (descarta clustering como análisis principal) |
| 2 | ¿Qué naturaleza tiene esa variable? | **Continua** (420–496 MW) | **Regresión** (descarta clasificación) |
| 3 | ¿Qué se quiere conocer? | El **valor numérico** de la potencia | Función de pérdida sobre la magnitud del error |
| 4 | ¿Qué pérdida y qué métrica? | Error cuadrático; RMSE en **MW** | Métrica con unidad física, interpretable por ingeniería |

## Por qué no clasificación

Podríamos *discretizar* `PE` (alta/media/baja potencia), pero eso:
- (a) obliga a fijar umbrales arbitrarios,
- (b) **destruye información** (un error de 1 MW y de 15 MW dentro de una misma clase serían iguales) y
- (c) **cambia la pregunta**: ya no cuántos MW, sino en qué rango. Si el interés operativo fuera una alarma ("¿cae por debajo de X MW?"), la clasificación sería una pregunta *derivada*, no esta.

## Por qué no clustering

El clustering **no usa una variable objetivo**: busca estructura en los datos. Aquí la variable objetivo existe y es la pregunta misma. El clustering sí podría servir como **análisis exploratorio complementario** (identificar regímenes de operación), pero no responde a la pregunta de investigación.

## Traducción al lenguaje algocrático

| Término tradicional | Término algocrático en este caso |
|---|---|
| Variable dependiente | Objetivo (*target*) `PE` |
| Variables independientes | Características (*features*) `AT, V, AP, RH` |
| Hipótesis H₀/H₁ | Hipótesis de rendimiento sobre el RMSE (arriba) |
| Diseño de investigación | Pipeline (Celdas 6–13) |

## Consecuencias prácticas de que sea regresión

- Métricas: **RMSE, MAE, R², sesgo** (no matriz de confusión ni ROC).
- Auditoría: **análisis de residuos** (homocedasticidad, normalidad aproximada, estructura no explicada).
- Partición: se compara la distribución del objetivo entre entrenamiento y prueba.

### 🔎 Celda 1 — Configuración, semillas y registro de auditoría

**Qué hace.** Importa las librerías, fija las **semillas** de aleatoriedad, define los **parámetros editables** del estudio, crea la carpeta de salidas, inicializa el **manifiesto de ejecución** (versiones, plataforma) y un **registro de auditoría** (`log`) donde cada paso deja una huella con fecha UTC.

**Por qué.** La reproducibilidad no se logra al final: se diseña desde la primera celda. Sin semilla, sin versiones y sin registro, nadie puede reproducir ni auditar el resultado.

**Qué observar.** El resumen impreso: versiones de Python y librerías, la semilla y el modo (rápido o completo). Si ves el aviso de `MODO_RAPIDO`, los resultados son solo una prueba de funcionamiento.

In [ ]:
# @title
# CELDA 1 · Configuración, semillas y registro de auditoría
import os, sys, re, json, hashlib, platform, subprocess, time, warnings, zipfile, random
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy import stats
import sklearn
import matplotlib
import matplotlib.pyplot as plt

try:
    from IPython.display import display, Markdown
except Exception:                       # entorno sin IPython (p. ej., pruebas locales)
    def display(*args, **kwargs):
        for a in args:
            print(a)
    def Markdown(texto):
        return texto

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------- Parámetros del estudio (editables) ----------
MODO_RAPIDO = False      # True = prueba de humo (menos repeticiones). NO usar para resultados del libro.
RANDOM_STATE = 42
TEST_SIZE = 0.20         # fracción (aprox.) reservada para el test sellado
N_SPLITS = 5             # folds de la validación cruzada
if MODO_RAPIDO:
    N_REPEATS, N_BOOT, N_ITER_BUSQUEDA, N_ITER_ANIDADA = 1, 300, 8, 4
    N_TREES, PERM_REPEATS, TRAMPA_R, TRAMPA_K = 60, 5, 4, 8
else:
    N_REPEATS, N_BOOT, N_ITER_BUSQUEDA, N_ITER_ANIDADA = 3, 2000, 30, 12
    N_TREES, PERM_REPEATS, TRAMPA_R, TRAMPA_K = 200, 30, 12, 20
EJECUTAR_TRAMPA = True   # Celda 18 (opcional)

# ---------- Semillas ----------
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
RNG = np.random.default_rng(RANDOM_STATE)

# ---------- Entorno y salidas ----------
EN_COLAB = "google.colab" in sys.modules
DIR_SALIDA = Path("/content/caso61_salidas") if EN_COLAB else Path("caso61_salidas")
DIR_SALIDA.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})

def guardar_fig(nombre, ajustar=True):
    """Guarda la figura actual en la carpeta de salidas y la muestra."""
    if ajustar:
        plt.tight_layout()
    ruta = DIR_SALIDA / nombre
    plt.savefig(ruta, dpi=200, bbox_inches="tight")
    plt.show()
    return ruta

# ---------- Registro de auditoría y manifiesto ----------
LOG_AUDITORIA = []
def log(evento, **datos):
    LOG_AUDITORIA.append({"t_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                          "evento": evento, **datos})

def a_json(o):
    """Convierte tipos de NumPy/Path/fecha para poder guardarlos en JSON."""
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.bool_,)):    return bool(o)
    if isinstance(o, np.ndarray):     return o.tolist()
    if isinstance(o, Path):           return str(o)
    if isinstance(o, datetime):       return o.isoformat()
    return str(o)

MANIFIESTO = {
    "inicio_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "plataforma": platform.platform(),
    "en_colab": EN_COLAB,
    "librerias": {"numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
                  "scikit-learn": sklearn.__version__, "matplotlib": matplotlib.__version__},
    "semilla": RANDOM_STATE,
    "modo_rapido": MODO_RAPIDO,
    "parametros": {"test_size": TEST_SIZE, "n_splits": N_SPLITS, "n_repeats": N_REPEATS, "n_boot": N_BOOT,
                   "n_iter_busqueda": N_ITER_BUSQUEDA, "n_iter_anidada": N_ITER_ANIDADA},
}
log("inicio", semilla=RANDOM_STATE, modo_rapido=MODO_RAPIDO)

print("Python", MANIFIESTO["python"], "|", MANIFIESTO["plataforma"])
for k, v in MANIFIESTO["librerias"].items():
    print(f"  {k:13s} {v}")
print("Semilla:", RANDOM_STATE, "| Carpeta de salidas:", DIR_SALIDA.resolve())
if MODO_RAPIDO:
    print("\n⚠️  MODO_RAPIDO activo: resultados NO válidos para el libro (solo prueba de funcionamiento).")

# Paso 1 · Identificar el dataset y perfilarlo

**Objetivo del paso.** Localizar los datos brutos, **documentar su origen** y registrar una fotografía inicial de su calidad *sin transformarlos*.
**Entregable clave.** Un **Perfil de datos** (tabla + figuras) que sirve como evidencia documental de la materia prima de la investigación.
**Referencia normativa (marco de lectura).** Medidas de calidad de datos para analítica y ML (familia ISO/IEC 5259) y consideraciones de sesgo (IEEE 7003-2024). *Verifica las cláusulas exactas en las normas antes de citarlas, pendiente con eso.*

### 🔎 Celda 2 — Localizar y leer el archivo

**Qué hace.** Busca en `/content` (y en la carpeta actual) un archivo `.xlsx`, `.xls`, `.csv` o `.zip`. Si no lo encuentra y estás en Colab, abre el **selector de archivos** para que lo subas. Si es un `.zip`, lo descomprime. Lee **todas las hojas** del libro de Excel y calcula la **huella SHA-256** del archivo.

**Por qué.** La huella permite demostrar, meses después, que se trabajó con *ese* archivo exacto y no con otro. Leer todas las hojas (no solo la primera) permite verificar en la siguiente celda si contienen datos distintos o repetidos.

**Qué observar.** Nombre del archivo, tamaño, huella, y una tabla con las hojas halladas y sus dimensiones. En la versión de UCI se espera una tabla de 9.568 filas por 5 columnas.

**Opcional.** Si ya subiste el archivo con otro nombre, escribe su ruta en `RUTA_ARCHIVO`.

In [ ]:
# @title
# CELDA 2 · Localizar y leer el archivo (Paso 1)
RUTA_ARCHIVO = None   # opcional: pega aquí la ruta exacta si ya subiste el archivo

def _sha256(ruta, bloque=1 << 20):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for chunk in iter(lambda: f.read(bloque), b""):
            h.update(chunk)
    return h.hexdigest()

def _candidatos():
    carpetas = [Path("/content"), Path.cwd()]
    vistos, cand = set(), []
    for carpeta in carpetas:
        if not carpeta.exists():
            continue
        for patron in ("*.xlsx", "*.xls", "*.csv", "*.zip"):
            for p in carpeta.glob(patron):
                rp = p.resolve()
                if rp in vistos or DIR_SALIDA.resolve() in rp.parents:
                    continue
                vistos.add(rp)
                cand.append(p)
    return sorted(cand, key=lambda p: p.stat().st_mtime, reverse=True)

def _elegir(cand):
    pref = [p for p in cand if re.search(r"ccpp|folds5x2|power|combined", p.name, re.I)]
    if pref:
        return pref[0]
    return cand[0] if cand else None

if RUTA_ARCHIVO is None:
    cand = _candidatos()
    if not cand and EN_COLAB:
        from google.colab import files
        print("No se encontró el archivo. Selecciónalo en la ventana que aparecerá...")
        files.upload()
        cand = _candidatos()
    ruta = _elegir(cand)
    if len(cand) > 1:
        print("Archivos candidatos (se elige el que coincide con CCPP, o el más reciente):")
        for p in cand:
            print("  -", p.name)
else:
    ruta = Path(RUTA_ARCHIVO)

if ruta is None or not Path(ruta).exists():
    raise FileNotFoundError("No encontré el dataset. Sube el archivo a /content o indica RUTA_ARCHIVO.")
ruta = Path(ruta)

if ruta.suffix.lower() == ".zip":
    carpeta_zip = ruta.with_suffix("")
    with zipfile.ZipFile(ruta) as z:
        z.extractall(carpeta_zip)
    internos = [p for p in carpeta_zip.rglob("*") if p.suffix.lower() in (".xlsx", ".xls", ".csv")]
    if not internos:
        raise FileNotFoundError("El .zip no contiene .xlsx, .xls ni .csv")
    ruta = sorted(internos)[0]

ext = ruta.suffix.lower()
if ext in (".xlsx", ".xls"):
    HOJAS_CRUDAS = pd.read_excel(ruta, sheet_name=None)       # dict {hoja: DataFrame}
elif ext == ".csv":
    HOJAS_CRUDAS = {"csv": pd.read_csv(ruta, sep=None, engine="python")}
else:
    raise ValueError(f"Formato no soportado: {ext}")

HUELLA_ARCHIVO = _sha256(ruta)
print("Archivo :", ruta.name)
print("Tamaño  :", round(ruta.stat().st_size / 1024, 1), "KB")
print("SHA-256 :", HUELLA_ARCHIVO)
display(pd.DataFrame({"filas": {h: d.shape[0] for h, d in HOJAS_CRUDAS.items()},
                      "columnas": {h: d.shape[1] for h, d in HOJAS_CRUDAS.items()}}))

MANIFIESTO["datos_origen"] = {"archivo": ruta.name, "sha256": HUELLA_ARCHIVO,
                              "hojas": {h: list(d.shape) for h, d in HOJAS_CRUDAS.items()}}
log("carga_archivo", archivo=ruta.name, sha256=HUELLA_ARCHIVO, hojas=list(HOJAS_CRUDAS))

### 🔎 Celda 3 — Estandarizar columnas y verificar hojas

**Qué hace.**
- (1) Renombra las columnas a los nombres canónicos `AT, V, AP, RH, PE` (aceptando variantes como `T` o `EP`) y detiene la ejecución con un mensaje claro si falta alguna.
- (2) Convierte todo a numérico y **cuenta los valores que no se pudieron convertir**.
- (3) Si el libro tiene varias hojas, **compara si son reordenaciones del mismo conjunto de filas**; si lo son, usar todas duplicaría los datos, así que se usa **solo una**.

**Por qué.** Usar dos hojas que contienen las mismas filas en distinto orden como si fueran datos independientes es un error clásico de **fuga y duplicación**. Aquí se verifica empíricamente en lugar de suponerlo.

**Qué observar.** La tabla de hojas: la columna *"mismas filas que la hoja 1"* te dice si son reordenaciones. Revisa también que `valores_no_numericos` sea 0.

In [ ]:
# CELDA 3 · Estandarizar columnas y verificar hojas (Paso 1)
COLUMNAS = ["AT", "V", "AP", "RH"]
OBJETIVO = "PE"
EQUIV = {"AT": "AT", "T": "AT", "TEMP": "AT", "TEMPERATURE": "AT",
         "V": "V", "VACUUM": "V", "EXHAUST VACUUM": "V",
         "AP": "AP", "PRESSURE": "AP", "AMBIENT PRESSURE": "AP",
         "RH": "RH", "HUMIDITY": "RH", "RELATIVE HUMIDITY": "RH",
         "PE": "PE", "EP": "PE", "ENERGY": "PE"}

def normalizar(df):
    ren = {c: EQUIV[str(c).strip().upper()] for c in df.columns if str(c).strip().upper() in EQUIV}
    out = df.rename(columns=ren)
    faltan = [c for c in COLUMNAS + [OBJETIVO] if c not in out.columns]
    if faltan:
        raise ValueError(f"Faltan columnas {faltan}. Columnas halladas: {list(df.columns)}")
    out = out[COLUMNAS + [OBJETIVO]].copy()
    n_antes = int(out.notna().sum().sum())
    out = out.apply(pd.to_numeric, errors="coerce")
    no_num = n_antes - int(out.notna().sum().sum())
    return out, no_num

HOJAS_NORM, filas_tabla = {}, []
for nombre, d in HOJAS_CRUDAS.items():
    dn, _ = normalizar(d)
    HOJAS_NORM[nombre] = dn
primera = next(iter(HOJAS_NORM))
ref = HOJAS_NORM[primera].sort_values(COLUMNAS + [OBJETIVO]).reset_index(drop=True)
for nombre, dn in HOJAS_NORM.items():
    ordenada = dn.sort_values(COLUMNAS + [OBJETIVO]).reset_index(drop=True)
    mismas = bool(ordenada.shape == ref.shape and np.allclose(ordenada.values, ref.values, equal_nan=True))
    filas_tabla.append({"hoja": nombre, "filas": len(dn), "mismas filas que la hoja 1": mismas})
tabla_hojas = pd.DataFrame(filas_tabla).set_index("hoja")
display(tabla_hojas)

HOJA_USADA = primera
df = HOJAS_NORM[HOJA_USADA].reset_index(drop=True)
_, NO_NUMERICOS = normalizar(HOJAS_CRUDAS[HOJA_USADA])
print(f"Hoja usada: '{HOJA_USADA}' | filas: {len(df)} | valores_no_numericos: {NO_NUMERICOS}")
if len(HOJAS_NORM) > 1 and not tabla_hojas["mismas filas que la hoja 1"].all():
    print("⚠️  Hay hojas con contenido distinto a la hoja 1. Se usa SOLO la hoja 1; revisa el origen de las otras.")
elif len(HOJAS_NORM) > 1:
    print("Las hojas son reordenaciones del mismo conjunto de filas: se usa SOLO una para no duplicar datos.")

MANIFIESTO["datos_origen"].update({"hoja_usada": HOJA_USADA, "n_filas": int(len(df)),
                                   "valores_no_numericos": int(NO_NUMERICOS)})
log("estandarizacion", hoja_usada=HOJA_USADA, n_filas=len(df), no_numericos=int(NO_NUMERICOS))

### 🔎 Celda 4 — Perfil de datos (el entregable del Paso 1)

**Qué hace.** Construye el **Perfil de datos**: completitud (nulos), **unicidad** (filas duplicadas y *conflictos*: mismas características con distinto objetivo), **validez de rango** frente a los rangos documentados, estadísticos descriptivos, **valores atípicos** (regla 1,5×IQR; se *señalan*, no se eliminan), correlaciones (Pearson y Spearman) y **VIF** (colinealidad). Genera tres figuras.

**Por qué.** Antes de modelar hay que saber qué materia prima se tiene. El VIF importa porque `AT` y `V` están fuertemente correlacionadas: eso afecta la interpretación de coeficientes y de importancias, no la capacidad predictiva.

**Qué observar.**
- *Nulos y duplicados*: ¿hay datos faltantes? ¿Filas repetidas? Los duplicados importarán en el Paso 3 (podrían caer en entrenamiento y prueba a la vez).
- *Conflictos*: grupos con mismas características pero distinto `PE` indican ruido de medición.
- *¿Dentro del rango documentado?*: cualquier `False` merece una explicación física.
- *VIF*: valores altos (>5 o >10) indican colinealidad.
- *Figura 1*: distribuciones y relación de cada variable con `PE`. *Figura 2*: correlaciones.

**Archivos que produce:** `perfil_datos.csv`, `fig1_distribuciones.png`, `fig2_relaciones.png`, `fig3_correlaciones.png`.

In [ ]:
# CELDA 4 · Perfil de datos (Paso 1)
from sklearn.linear_model import LinearRegression

RANGOS_DOC = {"AT": (1.81, 37.11), "V": (25.36, 81.56), "AP": (992.89, 1033.30),
              "RH": (25.56, 100.16), "PE": (420.26, 495.76)}      # según la documentación de UCI
UNIDADES = {"AT": "°C", "V": "cm Hg", "AP": "mbar", "RH": "%", "PE": "MW"}
cols = COLUMNAS + [OBJETIVO]

n_filas = len(df)
dup_exactos = int(df.duplicated().sum())
conflictos = int((df.dropna().groupby(COLUMNAS)[OBJETIVO].nunique() > 1).sum())

filas = []
for c in cols:
    x = df[c].dropna()
    q1, q3 = x.quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = int(((x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)).sum())
    lo, hi = RANGOS_DOC[c]
    filas.append({"Variable": c, "Unidad": UNIDADES[c], "Nulos %": round(100 * df[c].isna().mean(), 3),
                  "Únicos": int(x.nunique()), "Mín": x.min(), "Máx": x.max(), "Media": x.mean(),
                  "Mediana": x.median(), "Desv.": x.std(), "Asimetría": stats.skew(x),
                  "Curtosis": stats.kurtosis(x), "Atípicos IQR": n_out,
                  "Atípicos %": round(100 * n_out / len(x), 2),
                  "Rango doc.": f"{lo} – {hi}",
                  "¿Dentro del rango?": bool(x.min() >= lo - 0.01 and x.max() <= hi + 0.01)})
perfil = pd.DataFrame(filas).set_index("Variable")

def vif(frame, columnas):
    out = {}
    for c in columnas:
        otras = [k for k in columnas if k != c]
        r2 = LinearRegression().fit(frame[otras], frame[c]).score(frame[otras], frame[c])
        out[c] = np.inf if r2 >= 1 else 1.0 / (1.0 - r2)
    return pd.Series(out, name="VIF")

VIF = vif(df.dropna(), COLUMNAS)
CORR_P = df[cols].corr(method="pearson")
CORR_S = df[cols].corr(method="spearman")

print(f"Filas: {n_filas} | Nulos totales: {int(df.isna().sum().sum())} | "
      f"Duplicados exactos: {dup_exactos} | Grupos en conflicto (mismas X, distinto PE): {conflictos}")
display(perfil.round(3))
print("\nVIF (colinealidad entre predictoras):")
display(VIF.round(2).to_frame())

# ---- Figura 1: distribuciones ----
fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for ax, c in zip(axes, cols):
    ax.hist(df[c].dropna(), bins=40, color="#4C72B0", alpha=0.85)
    ax.set_title(f"{c} ({UNIDADES[c]})")
guardar_fig("fig1_distribuciones.png")

# ---- Figura 2: relación de cada predictora con PE (media por tramos) ----
fig, axes = plt.subplots(1, 4, figsize=(16, 3.4))
for ax, c in zip(axes, COLUMNAS):
    ax.scatter(df[c], df[OBJETIVO], s=3, alpha=0.15, color="#4C72B0")
    tramos = pd.qcut(df[c], 25, duplicates="drop")
    g = df.groupby(tramos, observed=True)
    ax.plot(g[c].mean(), g[OBJETIVO].mean(), color="#C44E52", lw=2, label="media por tramos")
    ax.set_xlabel(f"{c} ({UNIDADES[c]})"); ax.set_ylabel("PE (MW)")
axes[0].legend(frameon=False)
guardar_fig("fig2_relaciones.png")

# ---- Figura 3: correlaciones ----
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, M, titulo in zip(axes, [CORR_P, CORR_S], ["Pearson", "Spearman"]):
    im = ax.imshow(M.values, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols)
    ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols)
    ax.grid(False); ax.set_title(titulo)
    for i in range(len(cols)):
        for j in range(len(cols)):
            ax.text(j, i, f"{M.values[i, j]:.2f}", ha="center", va="center", fontsize=9)
fig.colorbar(im, ax=axes, shrink=0.8)
guardar_fig("fig3_correlaciones.png", ajustar=False)

perfil.to_csv(DIR_SALIDA / "perfil_datos.csv")
MANIFIESTO["perfil"] = {"n_filas": n_filas, "nulos_totales": int(df.isna().sum().sum()),
                        "duplicados_exactos": dup_exactos, "grupos_en_conflicto": conflictos,
                        "vif": VIF.round(3).to_dict(),
                        "fuera_de_rango": [c for c in cols if not bool(perfil.loc[c, "¿Dentro del rango?"])]}
log("perfil_datos", **MANIFIESTO["perfil"])

**Nota de procedencia (para el reporte).** El archivo no incluye marca de tiempo. Por eso: (i) no se puede verificar la **autocorrelación temporal** entre horas consecutivas; (ii) no se puede validar "hacia el futuro"; (iii) una partición aleatoria es la única opción disponible, y su optimismo potencial queda **declarado como limitación**, no ocultado.

# Paso 2 · Problema de investigación: función objetivo, métrica y criterios **pre-registrados**

**Objetivo del paso.** Convertir la pregunta de investigación en una **declaración formal y verificable**: qué se predice, con qué pérdida, con qué métrica y **qué resultado se considerará aceptable**, todo *antes* de ver el resultado en el conjunto de prueba.
**Entregable clave.** El **pre-registro**: un archivo JSON con huella SHA-256 y fecha.

### 🔎 Celda 5 — Pre-registro de la investigación

**Qué hace.** Declara en un diccionario la pregunta, el tipo de problema, el objetivo, la pérdida, la métrica primaria, las hipótesis, **la lista cerrada de modelos que se evaluarán en el test** y **cuatro criterios de aceptación con sus umbrales**:

- **C1 · Tolerancia de ingeniería:** RMSE de prueba ≤ 1 % de la potencia media de entrenamiento (parámetro `TOL_REL`, editable por el experto de dominio).
- **C2 · Mejora relativa:** el modelo candidato reduce el RMSE de prueba al menos un 20 % frente a la mejor línea base simple.
- **C3 · Estabilidad:** el coeficiente de variación del RMSE entre los folds externos de la CV anidada es ≤ 10 %.
- **C4 · Sin sesgo sistemático:** |error medio| ≤ 10 % del RMSE.

Luego guarda el JSON y calcula su **huella SHA-256**.

**Por qué.** Fijar el umbral *después* de ver el resultado permite ajustarlo para que el modelo "pase" (el equivalente algocrático del *p-hacking*). Aquí el umbral queda sellado antes. **El candidato también se declara a priori** (HistGradientBoosting con ajuste de hiperparámetros): no se elige "el que gane en CV".

**Qué observar.** El hash impreso. En un trabajo real, **publica ese hash** (repositorio con fecha, OSF, Zenodo) antes de evaluar el test: así demuestras que los criterios no se cambiaron después.

**Nota sobre quién fija el umbral.** `TOL_REL` es una decisión del **experto de dominio** (tolerancia de ingeniería), no del algoritmo. Es la parte humana de la significancia y la relevancia. Recuerda que parte de ser Curador es establecer métricas y auditar relevancias que tú mismo señalas.

In [ ]:
# CELDA 5 · Pre-registro (Paso 2)
TOL_REL = 0.01          # C1: RMSE_test <= TOL_REL * media(PE_train)
MEJORA_MIN = 0.20       # C2: mejora relativa mínima frente a la mejor línea base simple
CV_MAX_PCT = 10.0       # C3: CV% máximo del RMSE entre folds externos (CV anidada)
SESGO_FRAC = 0.10       # C4: |sesgo| <= SESGO_FRAC * RMSE_test

T_PRE_REGISTRO = datetime.now(timezone.utc)
PRE_REGISTRO = {
    "titulo": "Caso 6.1 - Prediccion de la potencia electrica neta horaria (CCPP)",
    "fecha_utc": T_PRE_REGISTRO.isoformat(timespec="seconds"),
    "pregunta_investigacion": "Predecir PE (MW) a partir de AT, V, AP y RH con error de ingenieria aceptable, y cuantificar su estabilidad e incertidumbre.",
    "tipo_problema": "regresion supervisada",
    "variable_objetivo": "PE (MW)",
    "variables_predictoras": COLUMNAS,
    "funcion_perdida_entrenamiento": "error cuadratico (MSE)",
    "metrica_primaria": "RMSE (MW) en el conjunto de prueba",
    "metricas_secundarias": ["MAE (MW)", "R2", "sesgo medio (MW)"],
    "hipotesis": {"H0": "El candidato no reduce el RMSE al menos MEJORA_MIN respecto de la mejor linea base simple.",
                  "H1": "El candidato reduce el RMSE al menos MEJORA_MIN y cumple la tolerancia de ingenieria."},
    "candidato_declarado_a_priori": "HistGradientBoostingRegressor con busqueda aleatoria de hiperparametros",
    "lineas_base_simples": ["Ridge lineal", "Ridge + Poly2"],
    "modelos_a_evaluar_en_test": ["Candidato", "Mejor linea base simple (segun CV en entrenamiento)", "Linea base trivial (media)"],
    "criterios_de_aceptacion": {
        "C1_tolerancia_ingenieria": {"regla": "RMSE_test <= tol_rel * media(PE_train)", "tol_rel": TOL_REL},
        "C2_mejora_relativa": {"regla": "1 - RMSE_candidato / RMSE_mejor_base_simple >= mejora_min", "mejora_min": MEJORA_MIN},
        "C3_estabilidad": {"regla": "CV% del RMSE entre folds externos de la CV anidada <= cv_max_pct", "cv_max_pct": CV_MAX_PCT},
        "C4_sesgo": {"regla": "|error medio en test| <= frac * RMSE_test", "frac": SESGO_FRAC},
    },
    "plan_estadistico": {"intervalos": f"bootstrap percentil 95% (B={N_BOOT})",
                         "comparacion_de_modelos": "bootstrap pareado sobre filas (no t-test sobre folds)",
                         "validacion": f"CV repetida {N_REPEATS}x{N_SPLITS} por grupos + CV anidada para el ajuste"},
    "protocolo_datos": {"semilla": RANDOM_STATE, "test_size": TEST_SIZE,
                        "particion": "por grupos de filas con caracteristicas identicas",
                        "test_sellado": "una sola evaluacion de la lista cerrada de modelos"},
    "huella_datos_sha256": HUELLA_ARCHIVO,
}
PRE_REGISTRO_JSON = json.dumps(PRE_REGISTRO, indent=2, ensure_ascii=False, sort_keys=True, default=a_json)
PRE_REGISTRO_HASH = hashlib.sha256(PRE_REGISTRO_JSON.encode("utf-8")).hexdigest()
(DIR_SALIDA / "pre_registro.json").write_text(PRE_REGISTRO_JSON, encoding="utf-8")

print("PRE-REGISTRO sellado el", PRE_REGISTRO["fecha_utc"], "(UTC)")
print("SHA-256 :", PRE_REGISTRO_HASH)
print("Criterios: C1 tol_rel =", TOL_REL, "| C2 mejora_min =", MEJORA_MIN,
      "| C3 cv_max_pct =", CV_MAX_PCT, "| C4 frac_sesgo =", SESGO_FRAC)
RESULTADOS = {}
MANIFIESTO["pre_registro"] = {"sha256": PRE_REGISTRO_HASH, "fecha_utc": PRE_REGISTRO["fecha_utc"]}
log("pre_registro", sha256=PRE_REGISTRO_HASH)

# Paso 3 · Seleccionar y preparar los datos

**Objetivo del paso.** Dividir los datos **antes de cualquier ajuste**, de modo que ninguna decisión de modelado haya visto el conjunto de prueba, y construir el esquema de validación cruzada.
**Principios.**
- (1) Toda transformación aprendida (imputación, escalado, polinomios) vive **dentro del pipeline**, para que se ajuste solo con datos de entrenamiento. -
- (2) Las filas con características idénticas se mantienen juntas en el mismo conjunto.
- (3) El test queda **sellado**.

### 🔎 Celda 6 — Partición por grupos y *test sellado*

**Qué hace.**
1. Descarta (y registra) filas con objetivo nulo, si las hubiera.
2. Asigna a cada fila un **grupo** según el *hash* de sus características: las filas **idénticas** (duplicados, o con distinto `PE`) quedan en el mismo grupo.
3. Reserva ≈20 % de los **grupos** para el test con `GroupShuffleSplit` (semilla fija) y **verifica que no haya grupos compartidos** entre entrenamiento y prueba.
4. Compara las distribuciones de entrenamiento y prueba (prueba de Kolmogorov-Smirnov y diferencia de medias en unidades de desviación) como **chequeo de sensatez** de la partición.
5. Calcula la **huella** de los índices de prueba y encierra el test en una **bóveda** (`BovedaTest`) que solo permite **una evaluación** de la lista de modelos pre-registrada.
6. Elimina de la memoria las tablas completas para que no se use el test por accidente.

**Por qué.** Si un duplicado cae en entrenamiento y en prueba, el modelo "reconoce" la fila y la métrica se infla (fuga de datos). La bóveda convierte la disciplina "no toques el test" en un mecanismo con registro. Es una **barrera de disciplina**, no de seguridad, entiéndelo así: Python permite saltársela, pero dejaría rastro en el registro de auditoría.

**Qué observar.** Tamaños de entrenamiento y prueba; la tabla KS (con ~9.500 filas, un *p* pequeño puede señalar diferencias mínimas: mira la columna `dif. en desviaciones`, que debería ser cercana a 0); la huella del test; el aviso "grupos compartidos: 0".

In [ ]:
# CELDA 6 · Partición por grupos y test sellado (Paso 3)
from sklearn.model_selection import GroupShuffleSplit

X_all = df[COLUMNAS].copy()
y_all = df[OBJETIVO].copy()

mask = y_all.notna().to_numpy()
N_DESCARTADAS = int((~mask).sum())
if N_DESCARTADAS:
    print(f"Se descartan {N_DESCARTADAS} filas con objetivo nulo (no se pueden usar para entrenar ni evaluar).")
    X_all, y_all = X_all.loc[mask].reset_index(drop=True), y_all.loc[mask].reset_index(drop=True)
log("filas_descartadas_objetivo_nulo", n=N_DESCARTADAS)

# Grupo = hash de las características (filas idénticas quedan juntas)
hash_filas = pd.util.hash_pandas_object(X_all.round(6).fillna(-999999.0), index=False).to_numpy()
grupo_id, _ = pd.factorize(hash_filas)

gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
idx_tr, idx_te = next(gss.split(X_all, y_all, groups=grupo_id))
grupos_compartidos = len(set(grupo_id[idx_tr]) & set(grupo_id[idx_te]))
assert grupos_compartidos == 0, "Hay grupos en entrenamiento y prueba: fuga de datos."
PARTICION_SIN_INTERSECCION = (grupos_compartidos == 0)

X_train = X_all.iloc[idx_tr].reset_index(drop=True)
y_train = y_all.iloc[idx_tr].reset_index(drop=True)
g_train = grupo_id[idx_tr]
MEDIA_Y_TRAIN = float(y_train.mean())

# ---- Chequeo de sensatez de la partición (solo distribuciones; no se toma ninguna decisión de modelado) ----
filas_ks = []
for c in COLUMNAS + [OBJETIVO]:
    a = X_train[c] if c in COLUMNAS else y_train
    b = X_all.iloc[idx_te][c] if c in COLUMNAS else y_all.iloc[idx_te]
    ks = stats.ks_2samp(a, b)
    filas_ks.append({"variable": c, "KS": ks.statistic, "p": ks.pvalue,
                     "dif. en desviaciones": (b.mean() - a.mean()) / a.std()})
tabla_ks = pd.DataFrame(filas_ks).set_index("variable")

# ---- Bóveda del test ----
class BovedaTest:
    """Disciplina de test sellado: permite UNA sola evaluación de una lista cerrada de modelos."""
    def __init__(self, X, y, huella):
        self._X, self._y, self.huella = X.copy(), y.copy(), huella
        self._usos, self.t_primer_uso = 0, None
    @property
    def usos(self):
        return self._usos
    @property
    def n(self):
        return len(self._y)
    def evaluar_modelos(self, modelos):
        if self._usos >= 1:
            raise RuntimeError("La bóveda de test ya fue abierta: un segundo uso contamina el test.")
        self._usos += 1
        self.t_primer_uso = datetime.now(timezone.utc)
        log("boveda_abierta", modelos=list(modelos))
        preds = {k: m.predict(self._X) for k, m in modelos.items()}
        return preds, self._y.to_numpy().copy()
    def X_para_analisis_posterior(self):
        if self._usos < 1:
            raise RuntimeError("Primero hay que abrir la bóveda con la evaluación pre-registrada.")
        return self._X.copy()

HUELLA_TEST = hashlib.sha256(np.sort(idx_te).astype(np.int64).tobytes()).hexdigest()
HUELLA_TRAIN = hashlib.sha256(np.sort(idx_tr).astype(np.int64).tobytes()).hexdigest()
BOVEDA = BovedaTest(X_all.iloc[idx_te], y_all.iloc[idx_te], HUELLA_TEST)
N_TRAIN, N_TEST = len(X_train), BOVEDA.n
del X_all, y_all, idx_te                      # el test ya solo es accesible a través de la bóveda

print(f"Entrenamiento: {N_TRAIN} filas ({len(np.unique(g_train))} grupos) | Test sellado: {N_TEST} filas")
print("Grupos compartidos entre entrenamiento y prueba:", grupos_compartidos)
print("Huella índices train:", HUELLA_TRAIN[:16], "... | test:", HUELLA_TEST[:16], "...")
display(tabla_ks.round(4))

MANIFIESTO["particion"] = {"n_train": N_TRAIN, "n_test": N_TEST, "grupos_compartidos": grupos_compartidos,
                           "huella_indices_train": HUELLA_TRAIN, "huella_indices_test": HUELLA_TEST,
                           "semilla": RANDOM_STATE}
log("particion", n_train=N_TRAIN, n_test=N_TEST, huella_test=HUELLA_TEST)

### 🔎 Celda 7 — Esquema de validación cruzada repetida **por grupos**

**Qué hace.** Define `particiones_cv`, una validación cruzada repetida (3 repeticiones × 5 folds) en la que **todas las filas de un mismo grupo caen en el mismo fold**. Verifica que, en cada repetición, los folds de validación **cubren todas las filas sin solaparse**.

**Por qué.** Implementarla a mano (en lugar de depender de una clase concreta de scikit-learn) garantiza el mismo comportamiento en cualquier versión de Colab. La repetición reduce la dependencia de una sola partición favorable o desfavorable.

**Qué observar.** El mensaje de verificación y el tamaño típico de cada fold. `CV_LISTA` contiene todas las particiones; `CV_UNA` es solo la primera repetición (una partición exacta, necesaria para predicciones *out-of-fold*).

In [ ]:
# CELDA 7 · Validación cruzada repetida por grupos (Paso 3)
def particiones_cv(grupos, n_splits, n_repeats, semilla):
    """CV repetida por grupos: filas con el mismo grupo caen siempre en el mismo fold."""
    rng = np.random.default_rng(semilla)
    ug = np.unique(grupos)
    out = []
    for _ in range(n_repeats):
        perm = rng.permutation(ug)
        for trozo in np.array_split(perm, n_splits):
            es_val = np.isin(grupos, trozo)
            out.append((np.where(~es_val)[0], np.where(es_val)[0]))
    return out

CV_LISTA = particiones_cv(g_train, N_SPLITS, N_REPEATS, RANDOM_STATE + 1)
CV_UNA = CV_LISTA[:N_SPLITS]            # primera repetición = partición exacta

ok_cv = True
for r in range(N_REPEATS):
    val = np.concatenate([CV_LISTA[r * N_SPLITS + k][1] for k in range(N_SPLITS)])
    ok_cv &= (len(val) == N_TRAIN) and (len(np.unique(val)) == N_TRAIN)
    for k in range(N_SPLITS):
        tr, va = CV_LISTA[r * N_SPLITS + k]
        ok_cv &= len(set(g_train[tr]) & set(g_train[va])) == 0
CV_SIN_FUGA = bool(ok_cv)
print("Particiones de CV:", len(CV_LISTA), f"({N_REPEATS} repeticiones x {N_SPLITS} folds)")
print("Cobertura completa sin solapes y sin grupos compartidos entre train/val:", CV_SIN_FUGA)
print("Tamaño medio de fold de validación:", int(np.mean([len(v) for _, v in CV_LISTA])), "filas")
assert CV_SIN_FUGA
log("cv_definida", n_particiones=len(CV_LISTA), sin_fuga=CV_SIN_FUGA)

# Paso 4 · Elegir arquitectura e hiperparámetros iniciales

**Objetivo del paso.** Elegir los modelos que se compararán y **justificar cada elección por escrito**. No existe un "mejor modelo" a priori (*no free lunch*): lo que se justifica es el **papel** de cada uno.
**Entregable clave.** Tabla de justificación + espacio de hiperparámetros declarado. Esto es clave para tu trabajo.

### 🔎 Celda 8 — Modelos, líneas base y espacio de búsqueda

**Qué hace.** Define cinco modelos, todos como **pipelines** (imputación → escalado → modelo) cuando corresponde:

| Modelo | Papel |
|---|---|
| **Dummy (media)** | Línea base **trivial**: predice siempre la media. Cualquier modelo que no la supere claramente no aprendió nada. |
| **Ridge lineal** | Línea base **simple e interpretable**. |
| **Ridge + Poly2** | Línea base simple con **no linealidad** de bajo costo (términos de grado 2). |
| **Random Forest** | Comparación no lineal, robusta, poco sensible a la escala. |
| **HistGB** | **Candidato declarado a priori**: boosting eficiente, captura no linealidad e interacciones. |

También declara el **espacio de hiperparámetros** del candidato (distribuciones log-uniformes y listas discretas) y construye la **tabla de justificación**.

**Por qué.** Las líneas base miden cuánto *valor agrega* el modelo complejo. Si un modelo lineal casi iguala al candidato, la complejidad no se justifica. Que el candidato se declare antes de ver resultados evita elegirlo por azar entre varios.

**Qué observar.** La tabla de justificación: cada fila responde *¿por qué este modelo?* y *¿qué riesgo tiene?*. Anota cómo cada decisión se conecta con el pre-registro.

In [ ]:
# CELDA 8 · Modelos, líneas base y espacio de búsqueda (Paso 4)
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from scipy.stats import loguniform

def _imp():
    return ("imputador", SimpleImputer(strategy="median"))

MODELOS = {
    "Dummy (media)": DummyRegressor(strategy="mean"),
    "Ridge lineal": Pipeline([_imp(), ("escalador", StandardScaler()), ("modelo", Ridge(alpha=1.0))]),
    "Ridge + Poly2": Pipeline([_imp(), ("escalador", StandardScaler()),
                               ("poly", PolynomialFeatures(degree=2, include_bias=False)),
                               ("escalador2", StandardScaler()), ("modelo", Ridge(alpha=1.0))]),
    "Random Forest": Pipeline([_imp(), ("modelo", RandomForestRegressor(n_estimators=N_TREES, n_jobs=1,
                                                                        random_state=RANDOM_STATE))]),
    "HistGB": Pipeline([_imp(), ("modelo", HistGradientBoostingRegressor(random_state=RANDOM_STATE))]),
}
LINEAS_BASE_SIMPLES = ["Ridge lineal", "Ridge + Poly2"]
CANDIDATO = "HistGB"

ESPACIO_HGB = {
    "modelo__learning_rate": loguniform(0.02, 0.3),
    "modelo__max_iter": [200, 400, 600],
    "modelo__max_leaf_nodes": [15, 31, 63],
    "modelo__min_samples_leaf": [10, 20, 40],
    "modelo__l2_regularization": loguniform(1e-4, 1.0),
}

tabla_justificacion = pd.DataFrame([
    ["Dummy (media)", "Línea base trivial", "Fija el piso: lo que se logra sin aprender nada", "Ninguna", "No sirve para predecir"],
    ["Ridge lineal", "Línea base simple", "Interpretable; coeficientes comparables tras escalar", "Estabilidad con colinealidad (AT–V)", "No captura curvatura ni interacciones"],
    ["Ridge + Poly2", "Línea base simple no lineal", "Curvatura de bajo costo, sigue siendo interpretable", "Más parámetros; riesgo leve de sobreajuste", "Extrapola mal fuera de rango"],
    ["Random Forest", "Comparación no lineal", "Robusto, sin supuestos de forma, poco sensible a la escala", "Menos preciso que boosting en datos suaves", "Costo de memoria; no extrapola"],
    ["HistGB (candidato)", "Candidato a priori", "Boosting eficiente; captura interacciones", "Requiere ajustar hiperparámetros", "Sobreajuste si no se controla; menos interpretable"],
], columns=["Modelo", "Rol", "¿Por qué?", "Fortaleza", "Riesgo"]).set_index("Modelo")
display(tabla_justificacion)
tabla_justificacion.to_csv(DIR_SALIDA / "tabla_justificacion.csv")
print("\nEspacio de hiperparámetros del candidato:")
for k_, d_ in {"learning_rate": "log-uniforme en [0.02, 0.3]", "max_iter": "{200, 400, 600}",
               "max_leaf_nodes": "{15, 31, 63}", "min_samples_leaf": "{10, 20, 40}",
               "l2_regularization": "log-uniforme en [1e-4, 1]"}.items():
    print(f"  {k_:18s} {d_}")
log("modelos_definidos", modelos=list(MODELOS), candidato=CANDIDATO)

# Paso 5 · Supervisar el entrenamiento y la validación

**Objetivo del paso.** Estimar el desempeño **sin tocar el test**, entender cómo se comporta el aprendizaje y **auditar** los errores.
**Entregables clave.** Tabla de CV, curvas de aprendizaje, estimación honesta (CV anidada) y diagnóstico de residuos.

### 🔎 Celda 9 — Validación cruzada repetida de los cinco modelos

**Qué hace.** Evalúa cada modelo con la CV repetida por grupos (15 particiones), midiendo RMSE, MAE y R² en validación **y en entrenamiento**. Resume con media, desviación, **CV%** (variabilidad relativa entre folds) y **brecha** train–validación. Dibuja el RMSE por fold y determina cuál es la **mejor línea base simple** entre las pre-registradas.

**Por qué.** Una sola partición puede ser engañosa; la repetición muestra la **variabilidad** del desempeño, que es información tan importante como su nivel. La brecha entre entrenamiento y validación es un indicador directo de sobreajuste.

**Qué observar.**
- *Dummy*: el piso. Todo debería superarlo con holgura.
- *CV%*: valores bajos = estable entre folds (relacionado con el criterio C3).
- *Brecha*: una brecha grande (entrenamiento mucho mejor que validación) delata sobreajuste.
- La caja de dispersión: cajas estrechas y separadas indican diferencias reales entre modelos.

In [ ]:
# CELDA 9 · Validación cruzada repetida (Paso 5)
from sklearn.model_selection import cross_validate

scoring = {"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error", "r2": "r2"}
RES_CV = {}
for nombre, modelo in MODELOS.items():
    t0 = time.time()
    r = cross_validate(modelo, X_train, y_train, cv=CV_LISTA, scoring=scoring,
                       return_train_score=True, n_jobs=-1)
    RES_CV[nombre] = {"rmse_val": -r["test_rmse"], "rmse_tr": -r["train_rmse"],
                      "mae_val": -r["test_mae"], "r2_val": r["test_r2"], "seg": time.time() - t0}
    print(f"  {nombre:15s} RMSE val = {RES_CV[nombre]['rmse_val'].mean():7.3f} MW   ({RES_CV[nombre]['seg']:.1f} s)")

tabla_cv = pd.DataFrame({
    k: {"RMSE val (MW)": v["rmse_val"].mean(), "± desv.": v["rmse_val"].std(ddof=1),
        "CV% RMSE": 100 * v["rmse_val"].std(ddof=1) / v["rmse_val"].mean(),
        "MAE val (MW)": v["mae_val"].mean(), "R² val": v["r2_val"].mean(),
        "Brecha train–val (MW)": (v["rmse_val"] - v["rmse_tr"]).mean()}
    for k, v in RES_CV.items()}).T.sort_values("RMSE val (MW)")
display(tabla_cv.round(3))
tabla_cv.to_csv(DIR_SALIDA / "tabla_cv.csv")

MEJOR_SIMPLE = min(LINEAS_BASE_SIMPLES, key=lambda k: RES_CV[k]["rmse_val"].mean())
MEJOR_CV = min([k for k in MODELOS if k != "Dummy (media)"], key=lambda k: RES_CV[k]["rmse_val"].mean())
print(f"\nMejor línea base simple (según CV): {MEJOR_SIMPLE}")
print(f"Mejor modelo en CV (solo informativo; el candidato fue declarado a priori: {CANDIDATO}): {MEJOR_CV}")

orden = list(tabla_cv.index)
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.boxplot([RES_CV[k]["rmse_val"] for k in orden])
ax.set_xticks(range(1, len(orden) + 1)); ax.set_xticklabels(orden, rotation=15)
ax.set_ylabel("RMSE en validación (MW)"); ax.set_title("RMSE por partición de CV repetida")
guardar_fig("fig4_cv_rmse.png")

RESULTADOS["cv"] = {k: {"rmse_media": float(v["rmse_val"].mean()), "rmse_sd": float(v["rmse_val"].std(ddof=1)),
                        "mae_media": float(v["mae_val"].mean()), "r2_media": float(v["r2_val"].mean())}
                    for k, v in RES_CV.items()}
RESULTADOS["mejor_linea_base_simple"] = MEJOR_SIMPLE
log("cv_repetida", mejor_simple=MEJOR_SIMPLE, mejor_cv=MEJOR_CV)

### 🔎 Celda 10 — Curvas de aprendizaje

**Qué hace.** Para la mejor línea base simple y para el candidato, entrena con porciones crecientes del entrenamiento (15 % → 100 %) y grafica el RMSE de **entrenamiento** y de **validación** (media ± desviación entre folds).

**Por qué.** Responde a la pregunta del comité "¿es suficiente el tamaño de muestra?" **con evidencia empírica** en lugar de una fórmula de muestreo: si la curva de validación ya se aplanó, más datos aportarían poco; si aún baja, más datos ayudarían.

**Qué observar.**
- Una **meseta** de la curva de validación = los datos alcanzan.
- Una **brecha persistente** entre las dos curvas = varianza alta (sobreajuste) o ruido irreducible.
- Si ambas curvas convergen a un valor alto = **sesgo alto** (el modelo es demasiado simple).

In [ ]:
# CELDA 10 · Curvas de aprendizaje (Paso 5)
from sklearn.model_selection import learning_curve

tamanos = np.linspace(0.15, 1.0, 6)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, nombre in zip(axes, [MEJOR_SIMPLE, CANDIDATO]):
    ts, tr_sc, va_sc = learning_curve(clone(MODELOS[nombre]), X_train, y_train, cv=CV_UNA,
                                      train_sizes=tamanos, scoring="neg_root_mean_squared_error", n_jobs=-1)
    tr_m, tr_s = (-tr_sc).mean(1), (-tr_sc).std(1)
    va_m, va_s = (-va_sc).mean(1), (-va_sc).std(1)
    ax.plot(ts, tr_m, "o-", color="#4C72B0", label="entrenamiento")
    ax.fill_between(ts, tr_m - tr_s, tr_m + tr_s, color="#4C72B0", alpha=0.2)
    ax.plot(ts, va_m, "o-", color="#C44E52", label="validación")
    ax.fill_between(ts, va_m - va_s, va_m + va_s, color="#C44E52", alpha=0.2)
    ax.set_title(nombre); ax.set_xlabel("tamaño de entrenamiento (filas)")
axes[0].set_ylabel("RMSE (MW)"); axes[0].legend(frameon=False)
guardar_fig("fig5_curvas_aprendizaje.png")
log("curvas_aprendizaje")

### 🔎 Celda 11 — Búsqueda de hiperparámetros y **validación cruzada anidada**

**Qué hace.**
1. **Búsqueda aleatoria** de hiperparámetros del candidato con CV por grupos; el mejor modelo se reajusta con **todo** el entrenamiento y queda listo para el test.
2. **CV anidada**: en cada fold externo se repite *todo el procedimiento* (búsqueda interna + reajuste) usando **solo** el entrenamiento de ese fold, y se evalúa en el fold externo. Esto estima el desempeño **de un procedimiento que incluye el ajuste**.
3. Calcula el **optimismo de selección**: la diferencia entre el RMSE anidado (honesto) y la mejor puntuación de la búsqueda (optimista).

**Por qué.** La mejor puntuación de una búsqueda es **optimista por construcción**: se eligió precisamente por ser la mejor en esos mismos folds (la "maldición del ganador"). Reportarla como estimación del desempeño sería inflar el resultado. La CV anidada es la estimación honesta; además produce el **CV%** del criterio C3.

**Qué observar.** Los cinco mejores conjuntos de hiperparámetros; el **optimismo** (suele ser pequeño pero positivo); y el RMSE anidado por fold. **Esta celda es la más lenta** del cuaderno así que espera un poco.

In [ ]:
# CELDA 11 · Búsqueda de hiperparámetros y CV anidada (Paso 5)
from sklearn.model_selection import RandomizedSearchCV

t0 = time.time()
busqueda = RandomizedSearchCV(clone(MODELOS[CANDIDATO]), ESPACIO_HGB, n_iter=N_ITER_BUSQUEDA, cv=CV_UNA,
                              scoring="neg_root_mean_squared_error", n_jobs=-1,
                              random_state=RANDOM_STATE, refit=True)
busqueda.fit(X_train, y_train)
modelo_candidato = busqueda.best_estimator_            # ya reajustado con TODO el entrenamiento
RMSE_BUSQUEDA_OPTIMISTA = float(-busqueda.best_score_)

res = pd.DataFrame(busqueda.cv_results_)
cols_param = [c for c in res.columns if c.startswith("param_")]
top = res.sort_values("rank_test_score").head(5)[["rank_test_score", "mean_test_score"] + cols_param].copy()
top["RMSE CV (MW)"] = -top.pop("mean_test_score")
top.columns = [c.replace("param_modelo__", "") for c in top.columns]
print(f"Búsqueda terminada en {time.time() - t0:.1f} s | mejor RMSE de la búsqueda (OPTIMISTA): {RMSE_BUSQUEDA_OPTIMISTA:.3f} MW")
display(top.round(4))

# ---- CV anidada: procedimiento completo dentro de cada fold externo ----
t0 = time.time()
rmse_anidado, params_anidados = [], []
for k, (tr, va) in enumerate(CV_UNA):
    Xtr, ytr, gtr = X_train.iloc[tr], y_train.iloc[tr], g_train[tr]
    cv_int = particiones_cv(gtr, 3, 1, RANDOM_STATE + 100 + k)
    bs = RandomizedSearchCV(clone(MODELOS[CANDIDATO]), ESPACIO_HGB, n_iter=N_ITER_ANIDADA, cv=cv_int,
                            scoring="neg_root_mean_squared_error", n_jobs=-1,
                            random_state=RANDOM_STATE + k, refit=True)
    bs.fit(Xtr, ytr)
    pred = bs.predict(X_train.iloc[va])
    rmse_anidado.append(float(np.sqrt(np.mean((y_train.iloc[va].to_numpy() - pred) ** 2))))
    params_anidados.append(bs.best_params_)
RMSE_ANIDADA = np.array(rmse_anidado)
RMSE_CAND_CV = float(RMSE_ANIDADA.mean())
CV_PCT_ANIDADA = float(100 * RMSE_ANIDADA.std(ddof=1) / RMSE_ANIDADA.mean())
OPTIMISMO = RMSE_CAND_CV - RMSE_BUSQUEDA_OPTIMISTA

print(f"\nCV anidada en {time.time() - t0:.1f} s")
print("RMSE por fold externo (MW):", np.round(RMSE_ANIDADA, 3))
print(f"RMSE anidado (HONESTO) = {RMSE_CAND_CV:.3f} MW | CV% = {CV_PCT_ANIDADA:.2f}%")
print(f"Optimismo de selección = {OPTIMISMO:+.3f} MW  (honesto − optimista de la búsqueda)")

RESULTADOS["busqueda"] = {"mejores_params": {k: (float(v) if isinstance(v, (np.floating, float)) else (int(v) if isinstance(v, (np.integer, int)) else v))
                                             for k, v in busqueda.best_params_.items()},
                          "rmse_busqueda_optimista": RMSE_BUSQUEDA_OPTIMISTA, "rmse_anidado": RMSE_CAND_CV,
                          "rmse_anidado_por_fold": RMSE_ANIDADA.tolist(), "cv_pct_anidada": CV_PCT_ANIDADA,
                          "optimismo": float(OPTIMISMO)}
log("busqueda_y_cv_anidada", rmse_anidado=RMSE_CAND_CV, optimismo=float(OPTIMISMO))

### 🔎 Celda 12 — Diagnóstico de residuos, comparación pareada y estado **preliminar** de los criterios

**Qué hace.**
1. Genera predicciones **fuera de muestra** (*out-of-fold*) del candidato y de la mejor línea base simple, y analiza los **residuos**: residuo vs. predicho, histograma, gráfico Q-Q y residuo vs. cada predictora. Calcula si el tamaño del error crece con la predicción (heterocedasticidad).
2. Compara candidato y línea base con **bootstrap pareado** sobre las filas: intervalo de confianza de la **diferencia de RMSE** y de la **mejora relativa**.
3. Evalúa de forma **preliminar** (solo con datos de entrenamiento) los criterios **C2** y **C3**.

**Por qué.**
- Los residuos revelan lo que la métrica agregada oculta: sesgos en regiones del espacio, error que crece con la potencia, estructura no capturada.
- Se usa **bootstrap pareado y no un t-test o Wilcoxon sobre los folds**: los folds comparten datos y **no son independientes**, por lo que esas pruebas subestiman la incertidumbre. (El Wilcoxon se reserva para comparar modelos en **varios conjuntos de datos**.)
- Ver el estado preliminar de los criterios *antes* del test permite anticipar el resultado sin contaminarlo.

**Qué observar.**
- *Residuos centrados en 0 y sin patrón* = buen ajuste; una curva sistemática indica estructura no capturada.
- *Q-Q*: puntos sobre la diagonal = errores aproximadamente normales; colas = errores grandes poco frecuentes.
- *Correlación |residuo|–predicho*: cerca de 0 = error de tamaño homogéneo.
- *IC de la mejora relativa*: si incluye valores por debajo del 20 %, el criterio C2 es frágil.

In [ ]:
# CELDA 12 · Residuos, comparación pareada y criterios preliminares (Paso 5)
from sklearn.model_selection import cross_val_predict

def metricas(y, p):
    e = p - y
    ss_res = float(np.sum(e ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    return {"RMSE": float(np.sqrt(np.mean(e ** 2))), "MAE": float(np.mean(np.abs(e))),
            "R2": 1.0 - ss_res / ss_tot, "Sesgo": float(np.mean(e))}

def bootstrap_pareado(y, preds, n_boot, semilla):
    """Remuestrea FILAS con los mismos índices para todos los modelos (diferencias pareadas)."""
    rng = np.random.default_rng(semilla)
    n = len(y)
    res = {k: {m: [] for m in ("RMSE", "MAE", "R2", "Sesgo")} for k in preds}
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        for k in preds:
            for m, v in metricas(y[idx], preds[k][idx]).items():
                res[k][m].append(v)
    return {k: {m: np.array(v) for m, v in d.items()} for k, d in res.items()}

def ic95(a):
    lo, hi = np.percentile(a, [2.5, 97.5])
    return float(lo), float(hi)

y_tr = y_train.to_numpy()
oof_cand = cross_val_predict(clone(modelo_candidato), X_train, y_train, cv=CV_UNA, n_jobs=-1)
oof_base = cross_val_predict(clone(MODELOS[MEJOR_SIMPLE]), X_train, y_train, cv=CV_UNA, n_jobs=-1)
res_oof = y_tr - oof_cand

rho, p_rho = stats.spearmanr(np.abs(res_oof), oof_cand)
print(f"Residuos OOF del candidato: media = {res_oof.mean():+.3f} MW | desv. = {res_oof.std(ddof=1):.3f} MW")
print(f"Correlación de Spearman |residuo| vs. predicho: {rho:+.3f} (p = {p_rho:.3g})")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
axes[0].scatter(oof_cand, res_oof, s=3, alpha=0.2, color="#4C72B0"); axes[0].axhline(0, color="#C44E52")
axes[0].set_xlabel("PE predicho (MW)"); axes[0].set_ylabel("residuo (MW)"); axes[0].set_title("Residuo vs. predicho")
axes[1].hist(res_oof, bins=50, color="#4C72B0", alpha=0.85); axes[1].set_xlabel("residuo (MW)"); axes[1].set_title("Distribución del residuo")
stats.probplot(res_oof, dist="norm", plot=axes[2]); axes[2].set_title("Gráfico Q-Q")
guardar_fig("fig6_residuos_oof.png")

fig, axes = plt.subplots(1, 4, figsize=(15, 3.2), sharey=True)
for ax, c in zip(axes, COLUMNAS):
    ax.scatter(X_train[c], res_oof, s=3, alpha=0.15, color="#4C72B0"); ax.axhline(0, color="#C44E52")
    ax.set_xlabel(c)
axes[0].set_ylabel("residuo (MW)")
guardar_fig("fig7_residuos_vs_predictoras.png")

# ---- Comparación pareada (bootstrap sobre filas, predicciones OOF) ----
BP = bootstrap_pareado(y_tr, {"cand": oof_cand, "base": oof_base}, N_BOOT, RANDOM_STATE + 5)
dif = BP["base"]["RMSE"] - BP["cand"]["RMSE"]
mej = 1 - BP["cand"]["RMSE"] / BP["base"]["RMSE"]
print(f"\nDiferencia de RMSE (base − candidato), OOF: {dif.mean():.3f} MW | IC95% = [{ic95(dif)[0]:.3f}, {ic95(dif)[1]:.3f}]")
print(f"Mejora relativa, OOF: {100 * mej.mean():.1f}% | IC95% = [{100 * ic95(mej)[0]:.1f}%, {100 * ic95(mej)[1]:.1f}%]")

# ---- Estado PRELIMINAR de C2 y C3 (solo entrenamiento) ----
rmse_base_cv = float(RES_CV[MEJOR_SIMPLE]["rmse_val"].mean())
mejora_cv = 1 - RMSE_CAND_CV / rmse_base_cv
prelim = pd.DataFrame([
    ["C2 mejora vs. mejor base simple (CV)", f"≥ {MEJORA_MIN:.0%}", f"{mejora_cv:.1%}", "✔" if mejora_cv >= MEJORA_MIN else "✘"],
    ["C3 estabilidad (CV% anidada)", f"≤ {CV_MAX_PCT:.0f}%", f"{CV_PCT_ANIDADA:.2f}%", "✔" if CV_PCT_ANIDADA <= CV_MAX_PCT else "✘"],
], columns=["Criterio", "Umbral pre-registrado", "Valor (solo entrenamiento)", "Estado"]).set_index("Criterio")
print("\nESTADO PRELIMINAR (basado únicamente en entrenamiento; el test sigue sellado):")
display(prelim)

RESULTADOS["preliminar"] = {"mejora_cv": float(mejora_cv), "ic95_mejora_oof": [float(x) for x in ic95(mej)],
                            "rho_abs_res_pred": float(rho)}
log("diagnostico_residuos_y_preliminar", mejora_cv=float(mejora_cv))

# Paso 6 · Interpretar los resultados y documentar el artefacto

**Objetivo del paso.** Evaluar **una sola vez** en el test sellado, decidir contra los criterios **pre-registrados**, entender *qué* usa el modelo para predecir y empaquetar todo de forma auditable.
**Entregables clave.** Resultados con IC, veredicto sobre C1–C4, interpretabilidad, reporte técnico, manifiesto, guion de defensa y paquete reproducible.

### 🔎 Celda 13 — Evaluación final en el test sellado (**una sola vez**)

**Qué hace.**
- (1) Reajusta la mejor línea base simple y la línea base trivial con todo el entrenamiento.
- (2) **Abre la bóveda una única vez** para obtener las predicciones de los tres modelos pre-registrados.
- (3) Calcula RMSE, MAE, R² y sesgo, con **intervalos de confianza del 95 %** por bootstrap sobre filas.
- (4) Calcula la **mejora relativa con su IC**.
- (5) Evalúa los cuatro criterios **C1–C4** contra los umbrales sellados en la Celda 5.
- (6) Grafica predicho vs. observado.

**Por qué.** Si se mira el test más de una vez para decidir algo, deja de ser una prueba independiente. Una sola evaluación, con una lista cerrada de modelos y criterios previos, hace que el resultado sea **creíble** aunque no sea el mejor posible. El intervalo de confianza responde a "¿cuánto confiar en este número?".

**Qué observar.**
- La tabla de métricas con IC: ¿el candidato supera claramente a las líneas base, o sus intervalos se solapan?
- La tabla de **criterios**: ✔ o ✘ por cada uno, con valor y umbral. Un ✘ **no se corrige**: se reporta y se discute (ese es el sentido del pre-registro).
- El gráfico predicho vs. observado: puntos cerca de la diagonal; abanicos o curvas indican errores sistemáticos.

In [ ]:
# CELDA 13 · Evaluación final en el test sellado (Paso 6)
modelo_simple = clone(MODELOS[MEJOR_SIMPLE]).fit(X_train, y_train)
modelo_dummy = clone(MODELOS["Dummy (media)"]).fit(X_train, y_train)

K_CAND = f"Candidato ({CANDIDATO} ajustado)"
K_SIMPLE = f"Línea base simple ({MEJOR_SIMPLE})"
K_DUMMY = "Línea base trivial (media)"
MODELOS_TEST = {K_CAND: modelo_candidato, K_SIMPLE: modelo_simple, K_DUMMY: modelo_dummy}

PREDS_TEST, Y_TEST = BOVEDA.evaluar_modelos(MODELOS_TEST)      # ← ÚNICO uso de la bóveda
print("Bóveda abierta. Usos:", BOVEDA.usos, "| filas de test:", BOVEDA.n)

M = {k: metricas(Y_TEST, p) for k, p in PREDS_TEST.items()}
BT = bootstrap_pareado(Y_TEST, PREDS_TEST, N_BOOT, RANDOM_STATE + 7)
filas = []
for k in PREDS_TEST:
    fila = {"Modelo": k}
    for m in ("RMSE", "MAE", "R2", "Sesgo"):
        lo, hi = ic95(BT[k][m])
        fila[m] = f"{M[k][m]:.3f}  [{lo:.3f}, {hi:.3f}]"
    filas.append(fila)
tabla_test = pd.DataFrame(filas).set_index("Modelo")
print(f"\nMétricas en el TEST (IC 95% por bootstrap, B = {N_BOOT}):")
display(tabla_test)
tabla_test.to_csv(DIR_SALIDA / "tabla_test.csv")

mejora_rel = 1 - BT[K_CAND]["RMSE"] / BT[K_SIMPLE]["RMSE"]
MEJORA_TEST = 1 - M[K_CAND]["RMSE"] / M[K_SIMPLE]["RMSE"]
IC_MEJORA = ic95(mejora_rel)
print(f"Mejora relativa del candidato sobre la mejor línea base simple: {MEJORA_TEST:.1%}  IC95% = "
      f"[{IC_MEJORA[0]:.1%}, {IC_MEJORA[1]:.1%}]")

# ---- Criterios pre-registrados ----
TOL_MW = TOL_REL * MEDIA_Y_TRAIN
rmse_c = M[K_CAND]["RMSE"]
sesgo_c = M[K_CAND]["Sesgo"]
CRITERIOS = [
    ("C1 Tolerancia de ingeniería", f"RMSE ≤ {TOL_MW:.2f} MW", f"{rmse_c:.3f} MW", bool(rmse_c <= TOL_MW)),
    ("C2 Mejora vs. mejor base simple", f"≥ {MEJORA_MIN:.0%}", f"{MEJORA_TEST:.1%}", bool(MEJORA_TEST >= MEJORA_MIN)),
    ("C3 Estabilidad (CV% anidada)", f"≤ {CV_MAX_PCT:.0f}%", f"{CV_PCT_ANIDADA:.2f}%", bool(CV_PCT_ANIDADA <= CV_MAX_PCT)),
    ("C4 Sin sesgo sistemático", f"|sesgo| ≤ {SESGO_FRAC:.0%} del RMSE", f"{abs(sesgo_c) / rmse_c:.1%} del RMSE", bool(abs(sesgo_c) <= SESGO_FRAC * rmse_c)),
]
tabla_criterios = pd.DataFrame([{"Criterio": a, "Umbral (pre-registrado)": b, "Valor observado": c,
                                 "Resultado": "✔ cumple" if d else "✘ no cumple"} for a, b, c, d in CRITERIOS]).set_index("Criterio")
print("\nVEREDICTO contra los criterios pre-registrados:")
display(tabla_criterios)
VEREDICTO = all(c[3] for c in CRITERIOS)
print("Veredicto global:", "H1 apoyada: se cumplen los 4 criterios" if VEREDICTO else
      "H1 NO apoyada en su totalidad: se reporta tal cual (los umbrales no se modifican después)")

fig, ax = plt.subplots(figsize=(5.2, 5))
ax.scatter(Y_TEST, PREDS_TEST[K_CAND], s=5, alpha=0.3, color="#4C72B0")
lim = [min(Y_TEST.min(), PREDS_TEST[K_CAND].min()), max(Y_TEST.max(), PREDS_TEST[K_CAND].max())]
ax.plot(lim, lim, color="#C44E52"); ax.set_xlabel("PE observado (MW)"); ax.set_ylabel("PE predicho (MW)")
ax.set_title("Test sellado: predicho vs. observado")
guardar_fig("fig8_predicho_vs_observado.png")

RESULTADOS["test"] = {"metricas": M, "ic95": {k: {m: ic95(BT[k][m]) for m in BT[k]} for k in BT},
                      "mejora_relativa": float(MEJORA_TEST), "ic95_mejora": list(IC_MEJORA),
                      "tolerancia_mw": float(TOL_MW), "media_y_train": MEDIA_Y_TRAIN,
                      "criterios": [{"criterio": a, "umbral": b, "valor": c, "cumple": d} for a, b, c, d in CRITERIOS],
                      "veredicto_global": bool(VEREDICTO)}
log("evaluacion_test", veredicto=bool(VEREDICTO), rmse_candidato=rmse_c)

### 🔎 Celda 14 — Análisis de errores en el test e **intervalos de predicción**

**Qué hace.**
1. Calcula el RMSE del candidato **por regímenes de operación** (terciles de temperatura y de humedad) con su tamaño de muestra.
2. Construye **intervalos de predicción del 90 %** con el método *split conformal*: usa los residuos *out-of-fold* del entrenamiento (nunca los del test) para fijar un margen, y **verifica la cobertura real** en el test.

**Por qué.** Un RMSE global puede esconder que el modelo falla justamente en un régimen de interés (por ejemplo, los días muy cálidos). Y un intervalo de predicción responde mejor que un solo número a la pregunta de ingeniería: *"¿entre qué valores esperamos la potencia?"*. Verificar su cobertura convierte la incertidumbre en algo **auditable**.

**Qué observar.**
- Si el RMSE sube mucho en algún tercil, el modelo es menos fiable en ese régimen.
- *Cobertura empírica* cercana al 90 % = la incertidumbre está bien calibrada. Una cobertura algo menor es esperable porque los residuos de calibración provienen de un modelo cuyos hiperparámetros se eligieron con esos mismos datos.
- Aquí se usa `X` del test **después** de la evaluación única, solo para describir errores, **no para cambiar el modelo**.

In [ ]:
# CELDA 14 · Errores por régimen e intervalos de predicción (Paso 6)
X_test_post = BOVEDA.X_para_analisis_posterior()
pred_c = PREDS_TEST[K_CAND]
err = pred_c - Y_TEST

filas = []
for var, etiquetas in [("AT", ["frío", "templado", "cálido"]), ("RH", ["seco", "medio", "húmedo"])]:
    terciles = pd.qcut(X_test_post[var], 3, labels=etiquetas)
    for et in etiquetas:
        m_ = (terciles == et).to_numpy()
        filas.append({"Variable": var, "Régimen": et, "n": int(m_.sum()),
                      "RMSE (MW)": float(np.sqrt(np.mean(err[m_] ** 2))), "Sesgo (MW)": float(np.mean(err[m_]))})
tabla_regimenes = pd.DataFrame(filas).set_index(["Variable", "Régimen"])
display(tabla_regimenes.round(3))
tabla_regimenes.to_csv(DIR_SALIDA / "tabla_regimenes.csv")

# ---- Intervalos de predicción (split conformal con residuos OOF del entrenamiento) ----
ALFA = 0.10
scores = np.sort(np.abs(y_tr - oof_cand))
n_cal = len(scores)
k_nivel = int(min(n_cal, np.ceil((n_cal + 1) * (1 - ALFA))))
Q_CONF = float(scores[k_nivel - 1])
dentro = (Y_TEST >= pred_c - Q_CONF) & (Y_TEST <= pred_c + Q_CONF)
COBERTURA = float(dentro.mean())
print(f"Intervalo de predicción {int((1 - ALFA) * 100)}%: predicción ± {Q_CONF:.2f} MW (ancho total {2 * Q_CONF:.2f} MW)")
print(f"Cobertura empírica en el test: {COBERTURA:.1%} (nominal {(1 - ALFA):.0%})")

fig, ax = plt.subplots(figsize=(9, 3.6))
orden_ = np.argsort(pred_c)[:: max(1, len(pred_c) // 250)]
ax.fill_between(range(len(orden_)), pred_c[orden_] - Q_CONF, pred_c[orden_] + Q_CONF, color="#4C72B0", alpha=0.25, label="intervalo 90%")
ax.plot(pred_c[orden_], color="#4C72B0", lw=1.2, label="predicción")
ax.scatter(range(len(orden_)), Y_TEST[orden_], s=6, color="#C44E52", label="observado")
ax.set_xlabel("muestra del test ordenada por predicción"); ax.set_ylabel("PE (MW)"); ax.legend(frameon=False, ncol=3)
guardar_fig("fig9_intervalos_prediccion.png")

RESULTADOS["incertidumbre"] = {"nivel": 1 - ALFA, "margen_mw": Q_CONF, "cobertura_test": COBERTURA}
RESULTADOS["regimenes"] = tabla_regimenes.reset_index().to_dict(orient="records")
log("errores_regimen_e_intervalos", cobertura=COBERTURA)

### 🔎 Celda 15 — Interpretabilidad: importancia por permutación, importancia **conjunta**, dependencia parcial y SHAP

**Qué hace.**
1. **Importancia por permutación** de cada variable (cuánto *empeora* el RMSE si se baraja esa columna).
2. **Importancia conjunta de AT + V**: se barajan *juntas*, porque están muy correlacionadas.
3. **Dependencia parcial** de la predicción respecto de cada variable (calculada sobre datos de entrenamiento).
4. **SHAP** (si se puede instalar): contribuciones por variable a las predicciones de una muestra del test.

**Por qué.** Que un modelo prediga bien no implica entender *qué* usa.

**Advertencia metodológica:** con variables correlacionadas, las importancias individuales **se reparten** y no se suman; barajar una sola columna deja a la otra "cubriendo" parte de la información. Por eso se calcula también la importancia del grupo. Todas estas técnicas describen **qué usa el modelo, no qué causa la potencia**: no son afirmaciones causales.

**Qué observar.**
- Si la importancia conjunta de AT+V supera a la suma de las individuales, hay **redundancia**.
- Las curvas de dependencia parcial deberían ser plausibles físicamente (por ejemplo, la potencia suele disminuir al aumentar la temperatura). Una forma inverosímil es una alerta.
- En SHAP, las variables ordenadas por contribución media absoluta; compáralas con la permutación (deberían coincidir en lo esencial).

In [ ]:
# CELDA 15 · Interpretabilidad (Paso 6)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

y_test_s = pd.Series(Y_TEST)
imp = permutation_importance(modelo_candidato, X_test_post, y_test_s, n_repeats=PERM_REPEATS,
                             random_state=RANDOM_STATE, scoring="neg_root_mean_squared_error", n_jobs=-1)
tabla_imp = pd.DataFrame({"aumento de RMSE (MW)": imp.importances_mean, "± desv.": imp.importances_std},
                         index=COLUMNAS).sort_values("aumento de RMSE (MW)", ascending=False)

def importancia_grupo(modelo, X, y, grupo, n_rep, semilla):
    rng = np.random.default_rng(semilla)
    base = float(np.sqrt(np.mean((y.to_numpy() - modelo.predict(X)) ** 2)))
    aumentos = []
    for _ in range(n_rep):
        Xp = X.copy()
        perm = rng.permutation(len(X))
        Xp[grupo] = X[grupo].to_numpy()[perm]
        aumentos.append(float(np.sqrt(np.mean((y.to_numpy() - modelo.predict(Xp)) ** 2))) - base)
    return float(np.mean(aumentos)), float(np.std(aumentos))

g_m, g_s = importancia_grupo(modelo_candidato, X_test_post, y_test_s, ["AT", "V"], PERM_REPEATS, RANDOM_STATE)
suma_ind = float(tabla_imp.loc[["AT", "V"], "aumento de RMSE (MW)"].sum())
print("Importancia por permutación (aumento de RMSE al barajar cada variable):")
display(tabla_imp.round(3))
print(f"\nImportancia CONJUNTA de AT + V: {g_m:.3f} ± {g_s:.3f} MW  |  suma de las individuales: {suma_ind:.3f} MW")
print("→ Conjunta > suma de individuales indica redundancia entre AT y V." if g_m > suma_ind else
      "→ No se observa redundancia marcada entre AT y V en este modelo.")

fig, ax = plt.subplots(figsize=(6.5, 3.2))
tabla_imp["aumento de RMSE (MW)"].iloc[::-1].plot.barh(xerr=tabla_imp["± desv."].iloc[::-1], ax=ax, color="#4C72B0")
ax.set_xlabel("aumento de RMSE (MW)"); ax.set_title("Importancia por permutación (test)")
guardar_fig("fig10_importancia_permutacion.png")

muestra_pdp = X_train.sample(min(1500, len(X_train)), random_state=RANDOM_STATE)
fig, ax = plt.subplots(figsize=(14, 3.2))
PartialDependenceDisplay.from_estimator(modelo_candidato, muestra_pdp, COLUMNAS, kind="average",
                                        grid_resolution=30, ax=ax, n_cols=4)
guardar_fig("fig11_dependencia_parcial.png")

# ---- SHAP (opcional) ----
SHAP_OK = False
try:
    try:
        import shap
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "shap"], check=False, capture_output=True)
        import shap
    fondo = X_train.sample(min(200, len(X_train)), random_state=RANDOM_STATE)
    muestra = X_test_post.sample(min(300, len(X_test_post)), random_state=RANDOM_STATE)
    explicador = shap.Explainer(modelo_candidato.predict, fondo)
    valores = explicador(muestra)
    plt.figure()
    shap.plots.beeswarm(valores, show=False)
    guardar_fig("fig12_shap_beeswarm.png")
    shap_medio = pd.Series(np.abs(valores.values).mean(0), index=COLUMNAS).sort_values(ascending=False)
    print("\nSHAP: contribución media absoluta (MW):")
    display(shap_medio.round(3).to_frame("|SHAP| medio"))
    SHAP_OK = True
except Exception as e:
    print("\nSHAP no disponible o falló (", type(e).__name__, "). Se continúa sin SHAP.")

RESULTADOS["interpretabilidad"] = {"permutacion": tabla_imp["aumento de RMSE (MW)"].to_dict(),
                                   "importancia_conjunta_AT_V": g_m, "suma_individuales_AT_V": suma_ind,
                                   "shap_disponible": SHAP_OK}
log("interpretabilidad", shap=SHAP_OK)

### 🔎 Celda 16 — Documentación: manifiesto, reporte técnico, guion de defensa y paquete

**Qué hace.** Convierte la ejecución en **artefactos auditables**, todos generados desde los resultados reales (nada escrito a mano):
- `manifiesto_ejecucion.json`: versiones, semilla, huellas de datos y de particiones, hash del pre-registro, uso de la bóveda, resultados y criterios.
- `registro_auditoria.json`: la bitácora cronológica de eventos.
- `requirements.txt`: el entorno exacto (`pip freeze`).
- `modelo_candidato.joblib`: el modelo entrenado.
- `reporte_tecnico.md`, `guion_defensa.md`, `traduccion_comite.md` y un `README`.
- Un `.zip` con todo (en Colab se ofrece la descarga).

**Por qué.** Un resultado sin su rastro no es auditable. Este paquete es lo que un revisor necesitaría para **reproducir y cuestionar** el trabajo. La tabla de **traducción** conecta la evidencia con las preguntas que el comité formulará en su propio lenguaje; el **guion de defensa** se escribe con los números reales.

**Qué observar.** Al final se muestran la tabla de traducción y el guion. Léelos como los leería un jurado: ¿cada afirmación tiene un número y una referencia en los artefactos? Revisa también la sección de **limitaciones** del reporte: debe ser honesta.

In [ ]:
# CELDA 16 · Documentación y artefactos (Paso 6)
import shutil
import joblib

RMSE_C, RMSE_S, RMSE_D = (M[K_CAND]["RMSE"], M[K_SIMPLE]["RMSE"], M[K_DUMMY]["RMSE"])
IC_RMSE_C = ic95(BT[K_CAND]["RMSE"])
IC_RMSE_S = ic95(BT[K_SIMPLE]["RMSE"])
R2_C = M[K_CAND]["R2"]
res_txt = "cumple los cuatro criterios pre-registrados" if VEREDICTO else \
          "no cumple todos los criterios pre-registrados (se reporta tal cual)"
crit_md = "\n".join(f"| {a} | {b} | {c} | {'✔' if d else '✘'} |" for a, b, c, d in CRITERIOS)

LIMITACIONES = [
    "El archivo no incluye marca de tiempo: no se verificó autocorrelación temporal ni se validó hacia el futuro; la partición aleatoria puede ser optimista.",
    "Datos de una sola planta, a plena carga y durante seis años: no generalizar a otras plantas, cargas parciales o condiciones fuera de los rangos observados.",
    "Asociación, no causalidad: AT y V están fuertemente correlacionadas; las importancias describen qué usa el modelo, no qué causa la potencia.",
    "Los intervalos de predicción usan residuos de un modelo cuyos hiperparámetros se eligieron con los mismos datos; la cobertura puede ser algo menor a la nominal.",
]
lim_md = "\n".join(f"- {t}" for t in LIMITACIONES)

reporte = f"""# Reporte técnico · Caso 6.1 (CCPP): regresión de la potencia eléctrica

## 1. Resumen
Se predijo la potencia neta horaria PE (MW) con variables ambientales (AT, V, AP, RH). El candidato ({CANDIDATO} ajustado) obtuvo, en el test sellado, un RMSE de **{RMSE_C:.3f} MW** (IC95% [{IC_RMSE_C[0]:.3f}, {IC_RMSE_C[1]:.3f}]), frente a {RMSE_S:.3f} MW de la mejor línea base simple ({MEJOR_SIMPLE}) y {RMSE_D:.3f} MW de la línea base trivial. La variación relativa del RMSE frente a la línea base simple fue de **{MEJORA_TEST:+.1%}** (positivo = reducción del error; IC95% [{IC_MEJORA[0]:+.1%}, {IC_MEJORA[1]:+.1%}]). El modelo {res_txt}.

## 2. Datos
- Origen: UCI Machine Learning Repository, *Combined Cycle Power Plant* (Tüfekci, 2014). Licencia CC BY 4.0.
- Archivo: `{MANIFIESTO['datos_origen']['archivo']}` · SHA-256: `{HUELLA_ARCHIVO}`.
- Filas usadas: {MANIFIESTO['datos_origen']['n_filas']} (hoja `{HOJA_USADA}`). Nulos: {MANIFIESTO['perfil']['nulos_totales']}. Duplicados exactos: {MANIFIESTO['perfil']['duplicados_exactos']}. Variables fuera de rango documentado: {MANIFIESTO['perfil']['fuera_de_rango'] or 'ninguna'}.
- Perfil completo: `perfil_datos.csv` y figuras 1–3.

## 3. Problema y pre-registro
Regresión supervisada. Pérdida: error cuadrático. Métrica primaria: RMSE (MW). Criterios y umbrales sellados el {PRE_REGISTRO['fecha_utc']} (UTC) con huella `{PRE_REGISTRO_HASH}`.

## 4. Método
- Partición por grupos (filas idénticas juntas): entrenamiento {N_TRAIN} filas; test sellado {N_TEST} filas (huella `{HUELLA_TEST[:16]}…`). Semilla {RANDOM_STATE}.
- Validación cruzada repetida {N_REPEATS}×{N_SPLITS} por grupos. Ajuste: búsqueda aleatoria ({N_ITER_BUSQUEDA} configuraciones) con **CV anidada** para la estimación honesta.
- Todas las transformaciones aprendidas están dentro de pipelines.

## 5. Resultados
| Modelo | RMSE test (MW) [IC95%] |
|---|---|
| {K_CAND} | {RMSE_C:.3f} [{IC_RMSE_C[0]:.3f}, {IC_RMSE_C[1]:.3f}] |
| {K_SIMPLE} | {RMSE_S:.3f} [{IC_RMSE_S[0]:.3f}, {IC_RMSE_S[1]:.3f}] |
| {K_DUMMY} | {RMSE_D:.3f} |

R² del candidato en test: {R2_C:.4f}. RMSE anidado (CV, honesto): {RMSE_CAND_CV:.3f} MW; optimismo de selección: {OPTIMISMO:+.3f} MW.

## 6. Criterios pre-registrados
| Criterio | Umbral | Valor | Resultado |
|---|---|---|---|
{crit_md}

## 7. Incertidumbre e interpretabilidad
- Intervalo de predicción del {int((1 - ALFA) * 100)}%: ± {Q_CONF:.2f} MW; cobertura empírica en test: {COBERTURA:.1%}.
- Importancia por permutación y dependencia parcial en las figuras 10–11 (describen qué usa el modelo, no causalidad). Importancia conjunta AT+V: {g_m:.3f} MW.

## 8. Limitaciones
{lim_md}

## 9. Reproducibilidad
Entorno en `requirements.txt` y `manifiesto_ejecucion.json`; semilla {RANDOM_STATE}; ejecutar el cuaderno con *Reiniciar y ejecutar todo*.

## 10. Referencias
Tüfekci, P. (2014). Prediction of full load electrical power output of a base load operated combined cycle power plant using machine learning methods. *International Journal of Electrical Power & Energy Systems, 60*, 126–140. https://doi.org/10.1016/j.ijepes.2014.02.027
"""

guion = f"""# Guion de defensa · Caso 6.1

*Estimado comité:* el trabajo no se sostiene en mi opinión sobre el modelo, sino en evidencia que pueden auditar.

1. **Los datos.** Provienen del repositorio UCI (Tüfekci, 2014), {MANIFIESTO['datos_origen']['n_filas']} registros horarios. El archivo está identificado por su huella SHA-256 y su perfil de calidad está documentado (nulos: {MANIFIESTO['perfil']['nulos_totales']}, duplicados exactos: {MANIFIESTO['perfil']['duplicados_exactos']}).
2. **El criterio de éxito se fijó antes de mirar el resultado.** El pre-registro (huella `{PRE_REGISTRO_HASH[:16]}…`) declara que el modelo debía tener un RMSE ≤ {TOL_MW:.2f} MW y mejorar al menos un {MEJORA_MIN:.0%} a la mejor línea base simple.
3. **El resultado.** En un conjunto de prueba sellado y evaluado una sola vez, el RMSE fue {RMSE_C:.3f} MW (IC95% [{IC_RMSE_C[0]:.3f}, {IC_RMSE_C[1]:.3f}]); la mejor línea base simple obtuvo {RMSE_S:.3f} MW. La variación relativa del RMSE fue de {MEJORA_TEST:+.1%} (positivo = mejora). El modelo {res_txt}.
4. **No dependo de una partición afortunada.** La estimación honesta del procedimiento completo (con ajuste de hiperparámetros) mediante validación cruzada anidada fue {RMSE_CAND_CV:.3f} MW, con una variabilidad entre folds de {CV_PCT_ANIDADA:.1f}%.
5. **Incertidumbre.** Un intervalo de predicción del {int((1 - ALFA) * 100)}% tiene ± {Q_CONF:.2f} MW y cubrió {COBERTURA:.1%} de los casos del test.
6. **Lo que NO afirmo.** No afirmo causalidad ni generalización a otras plantas o a cargas parciales; el archivo no tiene marca de tiempo y no pude verificar deriva temporal.
7. **Sobre la validez.** Si me preguntan por la validez de contenido, responderé con la calidad y cobertura del dato y sus rangos físicos; por la validez de criterio, con el desempeño en el test sellado; por la confiabilidad, con la estabilidad entre folds y la reproducibilidad por semilla y manifiesto. Es una traducción funcional de esos conceptos, no una equivalencia.
"""

traduccion = f"""| Lo que pide el comité | Evidencia que entrega este caso |
|---|---|
| ¿Cómo se operacionalizaron las variables? | Tabla de variables con unidad y rango documentado (perfil de datos) y pipeline de transformaciones declarado. |
| ¿Validez de contenido? | Perfil de calidad y cobertura del dato: completitud, unicidad, rangos físicos (traducción funcional, no equivalencia). |
| ¿Validez de criterio? | Desempeño en el test sellado: RMSE {RMSE_C:.3f} MW [{IC_RMSE_C[0]:.3f}, {IC_RMSE_C[1]:.3f}] frente a línea base. |
| ¿Confiabilidad? | Estabilidad entre folds (CV% anidada {CV_PCT_ANIDADA:.1f}%) y reproducibilidad por semilla y manifiesto. |
| ¿Tamaño de muestra suficiente? | Curvas de aprendizaje (fig. 5) e intervalos de confianza por bootstrap. |
| ¿Se controlaron las variables extrañas y el sesgo? | Partición por grupos sin fuga, pipeline, análisis de residuos por régimen, límites de generalización declarados. |
| ¿Juicio de expertos? | El experto de dominio fija la tolerancia de ingeniería (pre-registro) y revisa el análisis de errores por régimen. |
"""

(DIR_SALIDA / "reporte_tecnico.md").write_text(reporte, encoding="utf-8")
(DIR_SALIDA / "guion_defensa.md").write_text(guion, encoding="utf-8")
(DIR_SALIDA / "traduccion_comite.md").write_text(traduccion, encoding="utf-8")
(DIR_SALIDA / "README.md").write_text(
    "# Caso 6.1 · artefactos\n\nContenido: pre_registro.json, perfil_datos.csv, tablas *.csv, figuras fig*.png, "
    "reporte_tecnico.md, guion_defensa.md, traduccion_comite.md, manifiesto_ejecucion.json, registro_auditoria.json, "
    "requirements.txt, modelo_candidato.joblib.\n\nPara reproducir: instalar requirements.txt, subir el mismo archivo de datos "
    "(verificar SHA-256 en el manifiesto) y ejecutar el cuaderno completo.\n", encoding="utf-8")

joblib.dump(modelo_candidato, DIR_SALIDA / "modelo_candidato.joblib")
req = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
(DIR_SALIDA / "requirements.txt").write_text(req, encoding="utf-8")

MANIFIESTO["fin_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
MANIFIESTO["uso_boveda"] = {"usos": BOVEDA.usos, "primer_uso_utc": BOVEDA.t_primer_uso}
MANIFIESTO["resultados"] = RESULTADOS
MANIFIESTO["tiempos_cv_seg"] = {k: v["seg"] for k, v in RES_CV.items()}
(DIR_SALIDA / "manifiesto_ejecucion.json").write_text(json.dumps(MANIFIESTO, indent=2, ensure_ascii=False, default=a_json), encoding="utf-8")
(DIR_SALIDA / "registro_auditoria.json").write_text(json.dumps(LOG_AUDITORIA, indent=2, ensure_ascii=False, default=a_json), encoding="utf-8")

ruta_zip = shutil.make_archive(str(DIR_SALIDA.parent / "caso61_artefactos"), "zip", DIR_SALIDA)
print("Artefactos guardados en:", DIR_SALIDA.resolve())
print("Paquete comprimido:", ruta_zip)
log("artefactos_generados", zip=ruta_zip)

display(Markdown("### Tabla de traducción para el comité\n\n" + traduccion))
display(Markdown("### Guion de defensa (con los números de esta ejecución)\n\n" + guion))

if EN_COLAB:
    try:
        from google.colab import files
        files.download(ruta_zip)
    except Exception as e:
        print("No se pudo iniciar la descarga automática:", e)

### 🔎 Celda 17 — Lista de verificación de auditoría y prueba de reproducibilidad

**Qué hace.**
1. Verifica de forma **automática** doce condiciones del método (semilla, pre-registro anterior al test, bóveda abierta una sola vez, transformaciones dentro de pipelines, ausencia de fuga, líneas base, intervalos, CV anidada, criterios, artefactos...).
2. Ejecuta una **prueba de reproducibilidad**: reajusta el candidato desde cero con los mismos hiperparámetros y compara sus predicciones con las del test.

**Por qué.** Las buenas prácticas que no se comprueban se olvidan. Esta lista es un **mini-auditor** que revisa el propio cuaderno. La prueba de reproducibilidad responde a la pregunta del revisor: *"¿obtengo lo mismo si lo repito?"*.

**Qué observar.** Cada fila con ✔ o ✘. Un ✘ es un **hallazgo de auditoría**: explica qué falló y cómo corregirlo. En la reproducibilidad, la diferencia máxima debería ser prácticamente cero; si no lo es, el proceso no es determinista bit a bit (por ejemplo, por hilos de CPU o GPU) y eso debe documentarse con su tolerancia.

**Recomendación final.** Haz *Entorno de ejecución → Reiniciar y ejecutar todo* en una sesión limpia y confirma que se obtienen los mismos resultados.

In [ ]:
# CELDA 17 · Lista de verificación de auditoría y prueba de reproducibilidad (Paso 6)
archivos_esperados = ["pre_registro.json", "perfil_datos.csv", "manifiesto_ejecucion.json", "registro_auditoria.json",
                      "reporte_tecnico.md", "guion_defensa.md", "traduccion_comite.md", "requirements.txt",
                      "modelo_candidato.joblib"]
pipelines_ok = all(isinstance(m, Pipeline) for k, m in MODELOS.items() if k != "Dummy (media)")
checks = [
    ("1. Semilla global fijada y registrada", RANDOM_STATE is not None),
    ("2. Huella SHA-256 del archivo de datos registrada", bool(HUELLA_ARCHIVO)),
    ("3. Perfil de datos generado (nulos, duplicados, rangos, VIF)", (DIR_SALIDA / "perfil_datos.csv").exists()),
    ("4. Pre-registro creado ANTES de abrir el test sellado", bool(T_PRE_REGISTRO < BOVEDA.t_primer_uso)),
    ("5. Partición por grupos sin intersección train/test", bool(PARTICION_SIN_INTERSECCION)),
    ("6. Validación cruzada sin fuga entre folds", bool(CV_SIN_FUGA)),
    ("7. Transformaciones aprendidas dentro de pipelines", bool(pipelines_ok)),
    ("8. Línea base trivial y simple evaluadas junto al candidato", all(k in PREDS_TEST for k in (K_SIMPLE, K_DUMMY))),
    ("9. Bóveda del test abierta exactamente una vez", BOVEDA.usos == 1),
    ("10. CV anidada realizada (estimación honesta del ajuste)", len(RMSE_ANIDADA) == N_SPLITS),
    ("11. Intervalos de confianza por bootstrap reportados", all(len(v["RMSE"]) == N_BOOT for v in BT.values())),
    ("12. Criterios pre-registrados evaluados y artefactos completos", len(CRITERIOS) == 4 and all((DIR_SALIDA / a).exists() for a in archivos_esperados)),
]
tabla_checks = pd.DataFrame([{"Verificación": a, "Estado": "✔" if b else "✘"} for a, b in checks]).set_index("Verificación")
display(tabla_checks)
n_ok = sum(b for _, b in checks)
print(f"Resultado: {n_ok}/{len(checks)} verificaciones superadas.")

# ---- Prueba de reproducibilidad ----
modelo_repro = clone(modelo_candidato).fit(X_train, y_train)
pred_repro = modelo_repro.predict(BOVEDA.X_para_analisis_posterior())
dif_max = float(np.max(np.abs(pred_repro - PREDS_TEST[K_CAND])))
REPRODUCIBLE = dif_max < 1e-6
print(f"\nReproducibilidad: diferencia máxima entre predicciones = {dif_max:.3e} MW ->",
      "✔ idénticas dentro de la tolerancia (1e-6 MW)" if REPRODUCIBLE else
      "✘ no idénticas bit a bit: documenta la tolerancia y la causa probable (hilos, versión de librería)")
RESULTADOS["auditoria"] = {"verificaciones_ok": int(n_ok), "total": len(checks),
                           "reproducibilidad_dif_max": dif_max}
log("auditoria_final", ok=int(n_ok), reproducible=bool(REPRODUCIBLE))
(DIR_SALIDA / "registro_auditoria.json").write_text(json.dumps(LOG_AUDITORIA, indent=2, ensure_ascii=False, default=a_json), encoding="utf-8")

# Apéndice A · Trampa deliberada: la *maldición del ganador* (opcional)

### 🔎 Celda 18 — ¿Cuánto infla el resultado "mirar el test para elegir"?

**Qué hace.** Simula, **solo con datos de entrenamiento** (el test real no se toca), un error muy común: probar muchas configuraciones y quedarse con la que mejor puntúa en un conjunto pequeño de "vistazo", reportando esa puntuación. En cada repetición:
1. Divide el entrenamiento por grupos en *pseudo-entrenamiento*, un conjunto pequeño de *vistazo* (150 filas) y un conjunto *fresco*.
2. Entrena `K` configuraciones aleatorias del candidato y **elige la mejor según el vistazo**.
3. Compara el RMSE que *se reportaría* (en el vistazo) con el RMSE **honesto** de esa misma configuración en el conjunto fresco.
4. **Aísla el efecto de la selección**: el vistazo y el fresco son muestras distintas y pueden diferir en dificultad, lo que añade un desplazamiento que no es sesgo de selección. Se mide ese desplazamiento con el promedio de *todas* las configuraciones y se resta. Lo que queda es el **optimismo neto**: lo que se infla *solo por elegir al ganador*.

**Por qué.** Cuando se elige el mejor entre muchos con ruido de evaluación, el ganador tiende a estar entre los que tuvieron *buena suerte*: su puntuación es **optimista**. El efecto crece con el número de configuraciones comparadas y con lo pequeño que sea el conjunto de evaluación. Es la razón de ser del pre-registro, del test sellado y de la CV anidada.

**Qué observar.**
- La cifra **bruta** puede ser positiva, negativa o cero: mezcla el sesgo de selección con la diferencia de dificultad entre conjuntos, y fluctúa mucho entre repeticiones. Ese es, además, un aviso: *con un solo vistazo no se ve el sesgo*.
- El **optimismo neto** debería ser positivo en la mayoría de las repeticiones y con un error estándar pequeño.
- Con un test grande el efecto es modesto (centésimas de MW): lo importante es el **mecanismo**, que se agranda con menos datos o más comparaciones. Prueba a reducir `N_VISTAZO` o a aumentar `TRAMPA_K`.

In [ ]:
# CELDA 18 · Trampa deliberada: la maldición del ganador (opcional; no toca el test real)
if EJECUTAR_TRAMPA:
    from sklearn.model_selection import ParameterSampler
    N_VISTAZO = 150
    t0 = time.time()
    tam = pd.Series(g_train).value_counts()

    def una_repeticion(sem):
        rng = np.random.default_rng(sem)
        orden_g = rng.permutation(tam.index.to_numpy())
        cum = np.cumsum(tam.loc[orden_g].to_numpy())
        n = len(g_train)
        conj = np.where(cum <= 0.6 * n, 0, np.where(cum <= 0.6 * n + N_VISTAZO, 1, 2))
        mapa = dict(zip(orden_g, conj))
        asign = np.array([mapa[g] for g in g_train])
        Xa, ya = X_train[asign == 0], y_train[asign == 0]
        Xv, yv = X_train[asign == 1], y_train[asign == 1].to_numpy()
        Xf, yf = X_train[asign == 2], y_train[asign == 2].to_numpy()
        rv, rf = [], []
        for p in ParameterSampler(ESPACIO_HGB, TRAMPA_K, random_state=sem):
            m = clone(MODELOS[CANDIDATO]).set_params(**p).fit(Xa, ya)
            rv.append(float(np.sqrt(np.mean((yv - m.predict(Xv)) ** 2))))
            rf.append(float(np.sqrt(np.mean((yf - m.predict(Xf)) ** 2))))
        rv, rf = np.array(rv), np.array(rf)
        i = int(np.argmin(rv))                       # "ganador" según el vistazo
        return rv[i], rf[i], float(np.mean(rf - rv))  # (reportado, honesto, desplazamiento medio entre conjuntos)

    res_t = np.array([una_repeticion(RANDOM_STATE + 1000 + r) for r in range(TRAMPA_R)])
    reportado, honesto, desplaz = res_t[:, 0], res_t[:, 1], res_t[:, 2]
    bruto = honesto - reportado
    neto = bruto - desplaz                           # efecto de elegir al ganador, sin la dificultad relativa de los conjuntos
    se = lambda a: a.std(ddof=1) / np.sqrt(len(a))
    print(f"Repeticiones: {TRAMPA_R} | configuraciones por repetición (K): {TRAMPA_K} | "
          f"tamaño del vistazo: {N_VISTAZO} filas | {time.time() - t0:.0f} s")
    print(f"RMSE que se REPORTARÍA (mejor en el vistazo): {reportado.mean():.3f} MW")
    print(f"RMSE HONESTO de esa misma configuración (conjunto fresco): {honesto.mean():.3f} MW")
    print(f"Optimismo BRUTO = {bruto.mean():+.3f} MW (EE {se(bruto):.3f}): mezcla selección y dificultad relativa de los conjuntos")
    print(f"Optimismo NETO  = {neto.mean():+.3f} MW (EE {se(neto):.3f}) | positivo en {int((neto > 0).sum())} de {TRAMPA_R} repeticiones")
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    axes[0].boxplot([reportado, honesto]); axes[0].set_xticks([1, 2])
    axes[0].set_xticklabels(["reportado (vistazo)", "honesto (fresco)"]); axes[0].set_ylabel("RMSE (MW)")
    axes[0].set_title("Cifras brutas (ruidosas)")
    axes[1].boxplot([neto]); axes[1].axhline(0, color="#C44E52"); axes[1].set_xticks([1])
    axes[1].set_xticklabels(["optimismo neto"]); axes[1].set_ylabel("MW"); axes[1].set_title("Efecto de elegir al ganador")
    guardar_fig("fig13_maldicion_ganador.png")
    RESULTADOS["trampa"] = {"optimismo_bruto": float(bruto.mean()), "optimismo_neto": float(neto.mean()),
                            "ee_neto": float(se(neto)), "positivas_neto": int((neto > 0).sum()), "r": TRAMPA_R}
    log("trampa_maldicion_ganador", optimismo_neto=float(neto.mean()))
else:
    print("Celda 18 omitida (EJECUTAR_TRAMPA = False).")

---
# Cierre del caso: qué se entrega y qué sigue

## Lo que este caso demostró (en el lenguaje del libro)

| Paso | Evidencia producida |
|---|---|
| 1 · Dataset | Huella del archivo, perfil de calidad, hojas verificadas, límites de procedencia |
| 2 · Objetivo | Pre-registro sellado con hash: pregunta, métrica y umbrales **antes** del test |
| 3 · Preparación | Partición por grupos sin fuga, pipelines, test sellado con huella |
| 4 · Arquitectura | Tabla de justificación; candidato declarado a priori; líneas base |
| 5 · Validación | CV repetida, curvas de aprendizaje, CV anidada, residuos, comparación pareada |
| 6 · Resultado | Test evaluado una vez con IC, veredicto frente a criterios, incertidumbre, interpretabilidad, paquete auditable |

## Ejercicios para el lector

1. **Replica.** Cambia `RANDOM_STATE` (por ejemplo, a 7, 123 y 2024) y reporta cómo varían el RMSE de test, el optimismo y el veredicto. ¿Qué te dice sobre cuánto confiar en un solo número?
2. **Rompe.** Sustituye el pipeline por escalado aplicado a *todos* los datos antes de partir. Mide cuánto cambia la métrica. Con una sola variable y 9.500 filas el efecto puede ser pequeño: ¿es razonable concluir que "no importa"? ¿Qué cambia con 50 variables y 200 filas?
3. **Contamina.** Intenta abrir la bóveda por segunda vez. Lee el error y el registro de auditoría. ¿Qué rastro dejaría saltársela?
4. **Cambia el umbral.** Modifica `TOL_REL` *después* de ver el resultado y vuelve a ejecutar. ¿Qué cambió en el hash del pre-registro? ¿Qué diría un revisor?
5. **Audita.** Pide a un compañero que introduzca tres defectos en una copia del cuaderno (por ejemplo, usar el test para elegir hiperparámetros, quitar la semilla, reportar una sola corrida). Encuéntralos con la lista de verificación de la Celda 17 y redacta un informe de auditoría.
6. **Defiende.** Reescribe el guion de defensa en tus propias palabras para un comité de ciencias sociales: ¿qué evidencia de este caso responde a cada una de sus preguntas?

## Referencia del dataset

Tüfekci, P. (2014). Prediction of full load electrical power output of a base load operated combined cycle power plant using machine learning methods. *International Journal of Electrical Power & Energy Systems, 60*, 126–140. https://doi.org/10.1016/j.ijepes.2014.02.027 · Dataset: UCI Machine Learning Repository, *Combined Cycle Power Plant* (CC BY 4.0).